<h1 style="font-size:24px;">Cluster Nmuber</h1>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.validation import check_X_y
from itertools import combinations

# ----------------------------
# Load data
# ----------------------------
pre_path = "path/to/time_series_list_24_preCondRest.npy"
post_path = "path/to/time_series_list_24_postCondRest.npy"

time_series_list_pre = list(np.load(pre_path, allow_pickle=True))    # list of arrays: (T_pre_i, 24)
time_series_list_post = list(np.load(post_path, allow_pickle=True))  # list of arrays: (T_post_i, 24)

if len(time_series_list_pre) != len(time_series_list_post):
    raise ValueError(
        f"Mismatched number of subjects: pre={len(time_series_list_pre)}, "
        f"post={len(time_series_list_post)}"
    )

for i, (ts_pre, ts_post) in enumerate(zip(time_series_list_pre, time_series_list_post), 1):
    assert ts_pre.ndim == 2 and ts_pre.shape[1] == 24, \
        f"pre[{i}] shape={ts_pre.shape}, expected (*, 24)"
    assert ts_post.ndim == 2 and ts_post.shape[1] == 24, \
        f"post[{i}] shape={ts_post.shape}, expected (*, 24)"

# ----------------------------
# Preprocessing and stacking all frames
# ----------------------------
# Z-score options:
#   'time'  : z-score each ROI across time within each subject
#   'frame' : z-score each frame across ROIs within each subject
ZSCORE_MODE = 'time'   # choose from: 'time' or 'frame'

def safe_zscore(ts, axis=0):
    ts_z = zscore(ts, axis=axis, ddof=1)
    return np.nan_to_num(ts_z, nan=0.0, posinf=0.0, neginf=0.0)

X_list = []

# Pre
for ts in time_series_list_pre:
    if ts.shape[0] == 0:
        continue
    ts = np.asarray(ts, dtype=float)
    ts = safe_zscore(ts, axis=0 if ZSCORE_MODE == 'time' else 1)
    X_list.append(ts)

# Post
for ts in time_series_list_post:
    if ts.shape[0] == 0:
        continue
    ts = np.asarray(ts, dtype=float)
    ts = safe_zscore(ts, axis=0 if ZSCORE_MODE == 'time' else 1)
    X_list.append(ts)

if len(X_list) == 0:
    raise ValueError("No valid time series were found for CAP analysis.")

# Final frame matrix
X_cap = np.vstack(X_list)   # shape: (N_frames_kept, 24)
print("All frames stacked:", X_cap.shape)

# ----------------------------
# Ray-Turi index
# ----------------------------
def ray_turi(X, labels):
    X, labels = check_X_y(X, labels)
    le = LabelEncoder()
    labels = le.fit_transform(labels)

    n_samples, _ = X.shape
    n_labels = len(le.classes_)

    if n_labels < 2:
        raise ValueError("Number of distinct labels must be at least 2.")
    if n_samples < n_labels:
        raise ValueError("Number of samples must be >= number of labels.")

    intra_disp = 0.0
    cluster_means = []

    for k in range(n_labels):
        cluster_k = X[labels == k]
        mean_k = cluster_k.mean(axis=0)
        intra_disp += ((cluster_k - mean_k) ** 2).sum()
        cluster_means.append(mean_k)

    # Minimum squared distance between cluster centroids
    min_cluster_mean_diff = min(
        max(1e-10, ((mi - mj) ** 2).sum())
        for mi, mj in combinations(cluster_means, 2)
    )

    return intra_disp / (min_cluster_mean_diff * n_samples)

# ----------------------------
# Search over k and plot
# ----------------------------
range_n_clusters = range(2, 15)
rt_scores = []

for k in range_n_clusters:
    kmeans = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = kmeans.fit_predict(X_cap)
    score = ray_turi(X_cap, labels)
    rt_scores.append(score)
    print(f"k={k}, Ray-Turi index={score:.6f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(range_n_clusters), rt_scores, marker='o')
ax.set_xlabel("Number of clusters (k)", fontsize=12)
ax.set_ylabel("Ray-Turi Index (lower is better)", fontsize=12)
ax.set_title("Ray-Turi vs. k (CAP on scrubbed frames)", fontsize=13)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xticks(list(range_n_clusters))

plt.tight_layout()
plt.savefig("ray_turi_vs_k.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# =========================
# 1) Load data and preprocess
# =========================
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore
from sklearn.cluster import KMeans
from sklearn.metrics import davies_bouldin_score

pre_path = "path/to/time_series_list_24_preCondRest.npy"
post_path = "path/to/time_series_list_24_postCondRest.npy"

time_series_list_pre = list(np.load(pre_path, allow_pickle=True))    # list of arrays: (T_pre_i, 24)
time_series_list_post = list(np.load(post_path, allow_pickle=True))  # list of arrays: (T_post_i, 24)

if len(time_series_list_pre) != len(time_series_list_post):
    raise ValueError(
        f"Mismatched number of subjects: pre={len(time_series_list_pre)}, "
        f"post={len(time_series_list_post)}"
    )

# Check that each subject-level time series has 24 ROI columns
for i, (ts_pre, ts_post) in enumerate(zip(time_series_list_pre, time_series_list_post), 1):
    assert ts_pre.ndim == 2 and ts_pre.shape[1] == 24, \
        f"pre[{i}] shape={ts_pre.shape}, expected (*, 24)"
    assert ts_post.ndim == 2 and ts_post.shape[1] == 24, \
        f"post[{i}] shape={ts_post.shape}, expected (*, 24)"

# Z-score options:
#   'time'  : z-score each ROI across time within each subject
#   'frame' : z-score each frame across ROIs within each subject
ZSCORE_MODE = "time"   # choose from: "time" or "frame"

def safe_zscore(ts, axis=0):
    ts_z = zscore(ts, axis=axis, ddof=1)
    return np.nan_to_num(ts_z, nan=0.0, posinf=0.0, neginf=0.0)

X_list = []

for ts in time_series_list_pre:
    if ts.shape[0] == 0:
        continue
    ts = np.asarray(ts, dtype=float)
    ts = safe_zscore(ts, axis=0 if ZSCORE_MODE == "time" else 1)
    X_list.append(ts)

for ts in time_series_list_post:
    if ts.shape[0] == 0:
        continue
    ts = np.asarray(ts, dtype=float)
    ts = safe_zscore(ts, axis=0 if ZSCORE_MODE == "time" else 1)
    X_list.append(ts)

if len(X_list) == 0:
    raise ValueError("No valid time series were found for CAP analysis.")

X_cap = np.vstack(X_list)   # shape: (N_frames_kept, 24)
print("All frames stacked:", X_cap.shape)

# =========================
# 2) Davies-Bouldin index
# =========================
range_n_clusters = range(2, 15)
db_scores = []

for k in range_n_clusters:
    kmeans = KMeans(n_clusters=k, n_init=20, random_state=42)
    labels = kmeans.fit_predict(X_cap)
    score = davies_bouldin_score(X_cap, labels)
    db_scores.append(score)
    print(f"k={k}, DB index={score:.6f}")

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(range_n_clusters), db_scores, marker="o", linestyle="-", linewidth=1.8)
ax.set_xlabel("Number of clusters (k)", fontsize=12)
ax.set_ylabel("Davies-Bouldin Index (lower is better)", fontsize=12)
ax.set_title("Davies-Bouldin vs. k (CAP on scrubbed frames)", fontsize=13)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xticks(list(range_n_clusters))

# Mark the best k
best_idx = int(np.argmin(db_scores))
best_k = list(range_n_clusters)[best_idx]
best_val = db_scores[best_idx]
ax.scatter([best_k], [best_val], s=80, zorder=3)

# Optional annotation
# ax.annotate(
#     f"best k={best_k}\nDB={best_val:.3f}",
#     (best_k, best_val),
#     textcoords="offset points",
#     xytext=(10, -15)
# )

plt.tight_layout()
plt.savefig("davies_bouldin_vs_k.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
### <h1 style="font-size:36px;">Step 1: XXXX </h1>

<h2 style="font-size:24px;">HMM </h2>

In [ ]:
import numpy as np
from scipy.stats import zscore
from hmmlearn import hmm
from collections import defaultdict

# -----------------------------
pre_path  = "path/to//time_series_list_24_preCondRest.npy"
post_path = "path/to//time_series_list_24_postCondRest.npy"

time_series_list_pre  = list(np.load(pre_path,  allow_pickle=True))   # list of (T_pre_i, 24)
time_series_list_post = list(np.load(post_path, allow_pickle=True))   # list of (T_post_i, 24)

# 
n_pre, n_post = len(time_series_list_pre), len(time_series_list_post)
n_subj = min(n_pre, n_post)
time_series_list_pre  = time_series_list_pre[:n_subj]
time_series_list_post = time_series_list_post[:n_subj]

# 
for i, (ts_pre, ts_post) in enumerate(zip(time_series_list_pre, time_series_list_post), 1):
    assert ts_pre.ndim == 2 and ts_pre.shape[1] == 24, f"Subject {i} pre shape={ts_pre.shape}"
    assert ts_post.ndim == 2 and ts_post.shape[1] == 24, f"Subject {i} post shape={ts_post.shape}"

In [ ]:
import numpy as np
n_subj = len(time_series_list_pre)
subject_ids = list(range(n_subj))  
subject_ts_dict_pre = {i: np.asarray(ts) for i, ts in zip(subject_ids, time_series_list_pre)}
subject_ts_dict_post = {i: np.asarray(ts) for i, ts in zip(subject_ids, time_series_list_post)}

In [ ]:
import numpy as np
from scipy.stats import zscore
from hmmlearn import hmm
from collections import defaultdict

# -----------------------------
# 1) Prepare HMM training data
# -----------------------------
if len(time_series_list_pre) != len(time_series_list_post):
    raise ValueError(
        f"Mismatched number of subjects: pre={len(time_series_list_pre)}, "
        f"post={len(time_series_list_post)}"
    )

n_subj = len(time_series_list_pre)

X_list = []            # list of arrays to be concatenated for HMM input
lengths = []           # sequence length for each segment
subj_idx_per_seq = []  # subject index for each sequence
phase_per_seq = []     # 'pre' or 'post'

def safe_zscore(ts, axis=0):
    ts_z = zscore(ts, axis=axis, ddof=1)
    ts_z = np.nan_to_num(ts_z, nan=0.0, posinf=0.0, neginf=0.0)
    return ts_z

for s in range(n_subj):
    ts_pre = np.asarray(time_series_list_pre[s], dtype=float)
    ts_post = np.asarray(time_series_list_post[s], dtype=float)

    if ts_pre.ndim != 2 or ts_pre.shape[1] != 24:
        raise ValueError(f"Subject {s} pre shape={ts_pre.shape}, expected (*, 24)")
    if ts_post.ndim != 2 or ts_post.shape[1] != 24:
        raise ValueError(f"Subject {s} post shape={ts_post.shape}, expected (*, 24)")

    ts_pre_z = safe_zscore(ts_pre, axis=0)    
    ts_post_z = safe_zscore(ts_post, axis=0)  

    if ts_pre_z.shape[0] > 0:
        X_list.append(ts_pre_z)
        lengths.append(ts_pre_z.shape[0])
        subj_idx_per_seq.append(s)
        phase_per_seq.append("pre")

    if ts_post_z.shape[0] > 0:
        X_list.append(ts_post_z)
        lengths.append(ts_post_z.shape[0])
        subj_idx_per_seq.append(s)
        phase_per_seq.append("post")

if len(X_list) == 0:
    raise ValueError("No valid sequences were found for HMM training.")


X = np.vstack(X_list)   

# -----------------------------
# 2) Train HMM
# -----------------------------
k_states = 5

model = hmm.GaussianHMM(
    n_components=k_states,
    covariance_type="diag",
    n_iter=200,
    random_state=1,
    verbose=True
).fit(X, lengths)

state_seq = model.predict(X, lengths)

# -----------------------------
# 3) Split state sequence back into subject x phase
# -----------------------------
state_index_dict_pre = defaultdict(list)
state_index_dict_post = defaultdict(list)

offset = 0
for seq_len, subj_idx, phase in zip(lengths, subj_idx_per_seq, phase_per_seq):
    seq_states = state_seq[offset: offset + seq_len].tolist()
    offset += seq_len

    if phase == "pre":
        state_index_dict_pre[subj_idx].extend(seq_states)
    else:
        state_index_dict_post[subj_idx].extend(seq_states)

print(f"Number of subjects: {n_subj}")
print(
    f"Example subject 0: "
    f"pre length={len(state_index_dict_pre[0])}, "
    f"post length={len(state_index_dict_post[0])}"
)

<h1 style="font-size:24px;">OCR </h1>

In [ ]:
####################### Calculate the proportion #################################

In [ ]:
import pandas as pd
import numpy as np

def compute_subject_ocr(state_index_dict, k_clusters):
    """
    Compute subject-level state occupancy proportions.

    Supported input formats:
      1) subject -> list[int]
      2) subject -> {task_name: list[int], ...}
    """
    rows = []
    for subj, val in state_index_dict.items():
        all_states = []

        if isinstance(val, dict):                 # subject -> task -> list
            for states in val.values():
                all_states.extend(states)
        elif isinstance(val, (list, np.ndarray)): # subject -> list
            all_states.extend(val)
        else:
            raise TypeError(f"Unsupported value type for subject {subj}: {type(val)}")

        if len(all_states) == 0:
            continue

        counts = np.bincount(np.asarray(all_states, dtype=int), minlength=k_clusters)
        props  = counts / counts.sum()
        rows.append([subj, *props])

    cols = ["Subject"] + [f"State{i}_prop" for i in range(k_clusters)]
    return pd.DataFrame(rows, columns=cols)

# Usage
df_pre  = compute_subject_ocr(state_index_dict_pre,  k_clusters=k_states)
df_post = compute_subject_ocr(state_index_dict_post, k_clusters=k_states)

In [ ]:
# --------------------------------------------------
# Paired t-test and FDR correction across states
# --------------------------------------------------

import numpy as np
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import fdrcorrection

# Number of clusters (states)
k_clusters = k_states

# Compute subject-level occupancy rate (OCR)
df_pre  = compute_subject_ocr(state_index_dict_pre,  k_clusters=k_clusters)
df_post = compute_subject_ocr(state_index_dict_post, k_clusters=k_clusters)

# Convert to numpy arrays (remove subject column)
pre_array  = df_pre.iloc[:, 1:].to_numpy()
post_array = df_post.iloc[:, 1:].to_numpy()

# Paired t-tests for each state
t_vals = []
p_vals = []

for i in range(pre_array.shape[1]):
    t_stat, p_val = ttest_rel(pre_array[:, i], post_array[:, i])
    t_vals.append(t_stat)
    p_vals.append(p_val)

# FDR correction across states
reject, p_vals_corrected = fdrcorrection(p_vals, alpha=0.05)

# Display results
for i, (t_stat, p_corr, sig) in enumerate(zip(t_vals, p_vals_corrected, reject)):
    marker = "*" if sig else ""
    print(f"State {i}: t = {t_stat:.3f}, p_FDR = {p_corr:.6f} {marker}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def compute_block_state_counts(state_index_dict, target_state=0, window_size=5):
    """
    Compute block-wise counts of a target state for each subject.

    For each subject, the state sequence is divided into non-overlapping
    windows (blocks), and the number of occurrences of the target state
    is counted within each block.

    Parameters
    ----------
    state_index_dict : dict
        Mapping from subject ID to state sequence. Supported formats:
        1) subject -> list[int]
        2) subject -> np.ndarray
        3) subject -> dict of sequences
    target_state : int, default=0
        State label to count.
    window_size : int, default=5
        Number of TRs per block.

    Returns
    -------
    dict
        Dictionary mapping each subject to a 1D array of block-wise counts.
    """
    block_counts_dict = {}

    for subj, val in state_index_dict.items():
        if isinstance(val, dict):
            all_states = []
            for seq in val.values():
                all_states.extend(seq)
            seq = np.asarray(all_states, dtype=int)
        else:
            seq = np.asarray(val, dtype=int)

        T = len(seq)
        if T < window_size:
            block_counts_dict[subj] = np.array([], dtype=int)
            continue

        indicator = (seq == target_state).astype(int)
        n_blocks = T // window_size

        trimmed = indicator[: n_blocks * window_size]
        blocks = trimmed.reshape(n_blocks, window_size)
        counts = blocks.sum(axis=1)

        block_counts_dict[subj] = counts

    return block_counts_dict


def compute_group_block_curve(state_index_dict, target_state=0, window_size=5):
    """
    Compute the group-level block-wise curve for a target state.

    The group curve is obtained by averaging the block-wise counts across
    subjects for each block index. Only subjects with sufficient sequence
    length contribute to each block.

    Returns
    -------
    block_counts_dict : dict
        Subject-level block-wise counts.
    group_curve : np.ndarray
        Group-average block-wise counts.
    """
    block_counts_dict = compute_block_state_counts(
        state_index_dict,
        target_state=target_state,
        window_size=window_size
    )

    valid_blocks = [arr for arr in block_counts_dict.values() if len(arr) > 0]

    if len(valid_blocks) == 0:
        return block_counts_dict, np.array([])

    max_len = max(len(arr) for arr in valid_blocks)
    group_curve = []

    for i in range(max_len):
        vals = [arr[i] for arr in valid_blocks if len(arr) > i]
        group_curve.append(np.mean(vals))

    group_curve = np.asarray(group_curve)
    return block_counts_dict, group_curve


# Example usage
target_state = 0
window_size = 30

block_post_dict, group_curve_post = compute_group_block_curve(
    state_index_dict_post,
    target_state=target_state,
    window_size=window_size
)

print("Length of group_curve_post (number of blocks):", len(group_curve_post))
print("Counts in the first 10 blocks:", group_curve_post[:10])

x = np.arange(1, len(group_curve_post) + 1)

plt.figure(figsize=(10, 4))
plt.plot(x, group_curve_post, marker="o")
plt.xlabel(f"Block index (each = {window_size} TRs)")
plt.ylabel(f"Count of state {target_state} within block")
plt.title(
    f"Group-level block count curve for state {target_state} "
    f"(post, window = {window_size} TRs)"
)
plt.grid(True)
plt.tight_layout()
plt.show()

<h1 style="font-size:28px;">DT</h1>

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

def _accumulate_dwell_for_sequence(seq, k_clusters, collector):
    if seq is None or len(seq) == 0:
        return

    s = np.asarray(seq, dtype=int).tolist()

    curr = s[0]
    dwell = 1
    for x in s[1:]:
        if x == curr:
            dwell += 1
        else:
            collector[curr].append(dwell)
            curr = x
            dwell = 1
    collector[curr].append(dwell)

def compute_dwell_times(state_index_dict, k_clusters):
    """
    Compute dwell times for each subject and each state.

    Supported input formats:
      A) single-level: {subject: [state, state, ...]}
      B) nested:       {subject: {task: [state, state, ...]}}

    Returns
    -------
    dwell_times_dict : dict
        dwell_times_dict[subject][state] = [dwell1, dwell2, ...]
    """
    dwell_times_dict = defaultdict(lambda: defaultdict(list))

    for subj_id, val in state_index_dict.items():
        if isinstance(val, dict):
            for task, seq in val.items():
                _accumulate_dwell_for_sequence(seq, k_clusters, dwell_times_dict[subj_id])
        elif isinstance(val, (list, np.ndarray)):
            _accumulate_dwell_for_sequence(val, k_clusters, dwell_times_dict[subj_id])
        else:
            raise TypeError(f"Unsupported value type for subject {subj_id}: {type(val)}")

    return dwell_times_dict

def summarize_dwell(dwell_times_dict, k_clusters):
    state_subject_means = {s: [] for s in range(k_clusters)}

    for subj, per_state in dwell_times_dict.items():
        for s in range(k_clusters):
            dws = per_state.get(s, [])
            if len(dws) > 0:
                state_subject_means[s].append(float(np.mean(dws)))
            else:
                state_subject_means[s].append(0.0)

    print("=== Mean dwell time for each state ===")
    for s in range(k_clusters):
        arr = np.asarray(state_subject_means[s], dtype=float)
        if arr.size > 0:
            print(f"State {s}: mean = {arr.mean():.2f} ± {arr.std(ddof=1):.2f}, N_subjects = {arr.size}")
        else:
            print(f"State {s}: no data")

    return state_subject_means

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import fdrcorrection

k_clusters = k_states

dwell_pre = compute_dwell_times(state_index_dict_pre, k_clusters)
dwell_post = compute_dwell_times(state_index_dict_post, k_clusters)

print("\nPre:")
dwell_pre_avg = summarize_dwell(dwell_pre, k_clusters)

print("\nPost:")
dwell_post_avg = summarize_dwell(dwell_post, k_clusters)


def dwell_to_df(dwell_times_dict, k_clusters):
    rows = []
    for subj, per_state in dwell_times_dict.items():
        row = {"Subject": subj}
        for s in range(k_clusters):
            dws = per_state.get(s, [])
            row[f"State{s}_mean_dwell"] = float(np.mean(dws)) if len(dws) > 0 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


df_dwell_pre = dwell_to_df(dwell_pre, k_clusters)
df_dwell_post = dwell_to_df(dwell_post, k_clusters)


# Paired t-tests across states
t_vals = []
p_vals = []
state_cols = df_dwell_pre.columns[1:]

for col in state_cols:
    tval, pval = ttest_rel(df_dwell_pre[col], df_dwell_post[col])
    t_vals.append(tval)
    p_vals.append(pval)

# FDR correction across states
reject, p_vals_fdr = fdrcorrection(p_vals, alpha=0.05)

# Print results
for col, tval, pval, pval_fdr, sig in zip(state_cols, t_vals, p_vals, p_vals_fdr, reject):
    marker = "*" if sig else ""
    print(f"{col}: t = {tval:.3f}, p = {pval:.4f}, p_FDR = {pval_fdr:.4f} {marker}")

<h1 style="font-size:24px;">Transition Matrix </h1>

In [ ]:
######################## Transition Matrix ###########################

In [ ]:
import numpy as np

def compute_transition_matrices(state_index_dict, k_clusters):
    """
    Compute a k_clusters x k_clusters transition probability matrix
    for each subject.

    Input
    -----
    state_index_dict[subject] = [state sequence]

    Output
    ------
    subject_transition_matrices[subject] = transition probability matrix
    """
    subject_transition_matrices = {}

    for subj, state_seq in state_index_dict.items():
        transitions = np.zeros((k_clusters, k_clusters), dtype=int)

        for i in range(1, len(state_seq)):
            prev_state = state_seq[i - 1]
            curr_state = state_seq[i]
            transitions[prev_state, curr_state] += 1

        row_sums = transitions.sum(axis=1, keepdims=True)
        with np.errstate(divide="ignore", invalid="ignore"):
            trans_prob = np.divide(transitions, row_sums, where=row_sums != 0)

        subject_transition_matrices[subj] = trans_prob

    return subject_transition_matrices

In [ ]:
k_clusters = k_states

trans_pre = compute_transition_matrices(state_index_dict_pre, k_clusters)
trans_post = compute_transition_matrices(state_index_dict_post, k_clusters)

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import fdrcorrection
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# --------------------------------------------------
# 1) Paired t-tests on transition probabilities
#    with FDR correction across all k x k cells
# --------------------------------------------------

subject_ids = sorted(set(trans_pre.keys()) & set(trans_post.keys()))
trans_pre_array = np.stack([trans_pre[subj] for subj in subject_ids], axis=0)    # (N, k, k)
trans_post_array = np.stack([trans_post[subj] for subj in subject_ids], axis=0)  # (N, k, k)

raw_p_list = []
t_list = []
index_list = []

for i in range(k_clusters):
    for j in range(k_clusters):
        pre_vals = trans_pre_array[:, i, j]
        post_vals = trans_post_array[:, i, j]
        t_stat, p_val = ttest_rel(pre_vals, post_vals, nan_policy="omit")
        t_list.append(t_stat)
        raw_p_list.append(p_val)
        index_list.append((i, j))

# FDR correction
alpha = 0.05
reject, p_fdr_list = fdrcorrection(raw_p_list, alpha=alpha)

# Store corrected p-values in a matrix
p_fdr_matrix = np.full((k_clusters, k_clusters), np.nan)
annot_matrix = np.empty((k_clusters, k_clusters), dtype=object)

for idx, (i, j) in enumerate(index_list):
    p_fdr = p_fdr_list[idx]
    p_fdr_matrix[i, j] = p_fdr

    if p_fdr < 0.001:
        annot_matrix[i, j] = "***"
    elif p_fdr < 0.01:
        annot_matrix[i, j] = "**"
    elif p_fdr < alpha:
        annot_matrix[i, j] = "*"
    else:
        annot_matrix[i, j] = ""

# Group-level mean difference
diff_matrix = trans_pre_array.mean(axis=0) - trans_post_array.mean(axis=0)

# --------------------------------------------------
# 2) Bubble plot with FDR-corrected significance labels
# --------------------------------------------------

cmap = sns.diverging_palette(220, 20, as_cmap=True)

def bubble_transition_plot(
    diff_matrix,
    annot_matrix=None,
    title="",
    save_name="Trans_bubble_fdr.svg",
    cmap=cmap,
    max_bubble_area=650,
    min_bubble_area=20,
    grid_lw=1.2,
    bubble_edge_lw=0.6
):
    k = diff_matrix.shape[0]

    v = np.max(np.abs(diff_matrix)) if np.max(np.abs(diff_matrix)) > 0 else 1.0
    norm = Normalize(vmin=-v, vmax=v)

    xs, ys = np.meshgrid(np.arange(k), np.arange(k))
    x = xs.ravel()
    y = ys.ravel()
    vals = diff_matrix.ravel()

    absvals = np.abs(vals)
    if absvals.max() == 0:
        sizes = np.full_like(absvals, min_bubble_area, dtype=float)
    else:
        sizes = min_bubble_area + (max_bubble_area - min_bubble_area) * (absvals / absvals.max())

    colors = cmap(norm(vals))

    fig, ax = plt.subplots(figsize=(7, 6))

    ax.scatter(
        x, y,
        s=sizes,
        c=colors,
        edgecolors="white",
        linewidths=bubble_edge_lw
    )

    # Add significance labels
    if annot_matrix is not None:
        for i in range(k):
            for j in range(k):
                label = annot_matrix[i, j]
                if label != "":
                    ax.text(
                        j, i, label,
                        ha="center", va="center",
                        fontsize=11, fontweight="bold", color="black"
                    )

    ax.set_xlim(-0.5, k - 0.5)
    ax.set_ylim(k - 0.5, -0.5)
    ax.set_aspect("equal")

    ax.set_xticks(np.arange(k))
    ax.set_yticks(np.arange(k))

    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.set_xticks(np.arange(-0.5, k, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, k, 1), minor=True)
    ax.grid(which="minor", color="#BFBFBF", linestyle="-", linewidth=grid_lw)
    ax.tick_params(which="minor", bottom=False, left=False)

    ax.set_title(title, fontsize=14)

    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Pre - Post Probability Difference")

    plt.tight_layout()
    plt.savefig(save_name, format="svg", dpi=300, bbox_inches="tight")
    plt.show()

# Plot
bubble_transition_plot(
    diff_matrix,
    annot_matrix=annot_matrix,
    title="Pre - Post Transition Difference (FDR-corrected)",
    save_name="Trans_bubble_fdr.svg"
)

# --------------------------------------------------
# 3) Print significant transitions after FDR correction
# --------------------------------------------------

sig_indices_fdr = [(i, j) for (i, j), sig in zip(index_list, reject) if sig]
print(f"Number of significant transitions (FDR-corrected, q < {alpha}): {len(sig_indices_fdr)}")
print("Significant indices (from_state, to_state):", sig_indices_fdr)

for (i, j), t_stat, p_raw, p_fdr, sig in zip(index_list, t_list, raw_p_list, p_fdr_list, reject):
    marker = "*" if sig else ""
    print(f"Transition {i}->{j}: t = {t_stat:.3f}, p = {p_raw:.4f}, p_FDR = {p_fdr:.4f} {marker}")

<h1 style="font-size:24px;">Entropy </h1>

In [ ]:
import numpy as np
import scipy.linalg
from scipy.stats import ttest_rel

# =========================
# Entropy-related functions
# =========================
def as_stochastic(P, axis=1, eps=1e-12):
    """Normalize a matrix into a row-stochastic transition probability matrix."""
    P = np.asarray(P, dtype=float)
    P = np.clip(P, 0.0, 1.0)
    row_sums = P.sum(axis=axis, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return P / row_sums


def local_entropy(P, logbase=2):
    """Compute local entropy for each row of a transition matrix."""
    P = as_stochastic(P)
    log = np.log2 if logbase == 2 else np.log

    L = np.zeros_like(P)
    mask = P > 0
    L[mask] = log(P[mask])

    K = P @ L.T
    entropy_out = -np.diag(K)
    return entropy_out.reshape((P.shape[0], 1))


def stationary_distribution(P):
    """Compute the stationary distribution of a transition matrix."""
    P = as_stochastic(P)
    try:
        w, vl = scipy.linalg.eig(P, left=True, right=False)
        idx = np.argmin(np.abs(w - 1))
        v = np.real(vl[:, idx])
        mu = np.abs(v)
        s = mu.sum()
        if s == 0:
            raise RuntimeError
        mu /= s
    except Exception:
        mu = np.ones(P.shape[0]) / P.shape[0]
        for _ in range(2000):
            mu_new = mu @ P
            if np.linalg.norm(mu_new - mu, ord=1) < 1e-12:
                break
            mu = mu_new
    return mu


def trajectory_entropy(P, logbase=2, ridge=1e-8):
    """
    Compute the trajectory entropy matrix H for a transition matrix P.
    """
    P = as_stochastic(P)
    n = P.shape[0]
    mu = stationary_distribution(P)
    A = np.tile(mu, (n, 1))
    l_entropy = local_entropy(P, logbase=logbase)

    H_star = np.tile(l_entropy, (1, n))
    entropy_rate = float(mu @ l_entropy.flatten())

    mu_safe = mu.copy()
    mu_safe[mu_safe == 0] = 1e-12
    H_delta = np.diag(entropy_rate / mu_safe)

    M = np.eye(n) - P + A
    try:
        Minv = np.linalg.inv(M)
    except np.linalg.LinAlgError:
        Minv = np.linalg.inv(M + ridge * np.eye(n))

    K = Minv @ (H_star - H_delta)
    K_diag = np.diag(K).reshape(-1, 1)
    K_tilda = np.tile(K_diag.T, (n, 1))

    H = K - K_tilda + H_delta
    return H


def normalized_trajectory_entropy(P, logbase=2):
    """
    Compute the normalized trajectory entropy matrix.
    H_norm = H / entropy_rate
    """
    H = trajectory_entropy(P, logbase=logbase)

    mu = stationary_distribution(P)
    h_row = local_entropy(P, logbase=logbase).flatten()
    H_rate = float(mu @ h_row)

    if H_rate == 0:
        return np.zeros_like(H)

    return H / H_rate


# =========================
# Convert dict to aligned arrays
# =========================
def stack_subject_transition(trans_pre, trans_post):
    common = sorted(set(trans_pre.keys()) & set(trans_post.keys()))
    P_pre = np.stack([as_stochastic(trans_pre[s]) for s in common], axis=0)
    P_post = np.stack([as_stochastic(trans_post[s]) for s in common], axis=0)
    return P_pre, P_post, common


P_pre, P_post, subject_ids = stack_subject_transition(trans_pre, trans_post)
N, k, _ = P_pre.shape
print(f"Stacked: N = {N}, k = {k}")

# =========================
# Subject-level scalar trajectory entropy: mean(H)
# =========================
pre_scalar = np.empty(N, dtype=float)
post_scalar = np.empty(N, dtype=float)

for i in range(N):
    Hpre = trajectory_entropy(P_pre[i], logbase=2)
    Hpost = trajectory_entropy(P_post[i], logbase=2)
    pre_scalar[i] = Hpre.mean()
    post_scalar[i] = Hpost.mean()

# Paired t-test on the scalar mean(H)
tval, pval = ttest_rel(post_scalar, pre_scalar)
print(f"[Scalar mean(H)] paired t-test: t = {tval:.3f}, p = {pval:.4g}")

# =========================
# Subject-level trajectory entropy matrices: H (N, k, k)
# =========================
# pre_H = np.stack([trajectory_entropy(P_pre[i], logbase=2) for i in range(N)], axis=0)
# post_H = np.stack([trajectory_entropy(P_post[i], logbase=2) for i in range(N)], axis=0)

pre_H = np.stack([normalized_trajectory_entropy(P_pre[i]) for i in range(N)], axis=0)
post_H = np.stack([normalized_trajectory_entropy(P_post[i]) for i in range(N)], axis=0)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable


def plot_t_matrix_bubble(
    pre_H,
    post_H,
    title="Paired t-statistic Matrix",
    save_name="T_matrix.svg",
    cmap=None,
    max_size=650,
    min_size=30
):
    """
    Plot a bubble matrix of paired t-statistics comparing pre_H and post_H.

    Parameters
    ----------
    pre_H : np.ndarray
        Array of shape (N, k, k) for the pre condition.
    post_H : np.ndarray
        Array of shape (N, k, k) for the post condition.
    title : str
        Figure title.
    save_name : str
        Output file name for the saved figure.
    cmap : matplotlib colormap, optional
        Colormap used for the bubble colors.
    max_size : float
        Maximum bubble size.
    min_size : float
        Minimum bubble size.

    Returns
    -------
    t_mat : np.ndarray
        Matrix of paired t-statistics.
    p_mat : np.ndarray
        Matrix of raw p-values.
    """
    if cmap is None:
        cmap = sns.diverging_palette(220, 20, as_cmap=True)

    N, k, _ = pre_H.shape

    t_mat = np.zeros((k, k))
    p_mat = np.zeros((k, k))

    # Compute paired t-tests
    for i in range(k):
        for j in range(k):
            t, p = ttest_rel(pre_H[:, i, j], post_H[:, i, j], nan_policy="omit")
            t_mat[i, j] = t
            p_mat[i, j] = p

    # Color scaling
    vmax = np.percentile(np.abs(t_mat), 98)
    vmax = vmax if vmax > 0 else 1
    norm = Normalize(vmin=-vmax, vmax=vmax)

    # Coordinates
    xs, ys = np.meshgrid(np.arange(k), np.arange(k))
    x = xs.ravel()
    y = ys.ravel()
    vals = t_mat.ravel()

    # Bubble sizes
    absvals = np.abs(vals)
    if absvals.max() == 0:
        sizes = np.full_like(absvals, min_size, dtype=float)
    else:
        sizes = min_size + (max_size - min_size) * (absvals / absvals.max())

    colors = cmap(norm(vals))

    # Plot
    fig, ax = plt.subplots(figsize=(7, 6))

    ax.scatter(
        x, y,
        s=sizes,
        c=colors,
        edgecolors="white",
        linewidths=0.7
    )

    ax.set_xlim(-0.5, k - 0.5)
    ax.set_ylim(k - 0.5, -0.5)
    ax.set_aspect("equal")

    ax.set_xticks(np.arange(k))
    ax.set_yticks(np.arange(k))
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title(title, fontsize=14)

    # Grid
    ax.set_xticks(np.arange(-0.5, k, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, k, 1), minor=True)
    ax.grid(which="minor", color="#BFBFBF", linewidth=1.2)
    ax.tick_params(which="minor", bottom=False, left=False)

    # Colorbar
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("t statistic")

    plt.tight_layout()
    plt.savefig(save_name, format="svg", dpi=300, bbox_inches="tight")
    plt.show()

    return t_mat, p_mat

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
from nilearn.connectome import sym_matrix_to_vec

# ------------  取 state = 3 的 FC 并求 Δ(post - pre) ------------
# 形状: [N_subj × 442 × 442]
delta_G1 = fc_post_G1[1] - fc_pre_G1[1]      # 组1 
delta_G2 = fc_post_G2[1] - fc_pre_G2[1]      # 组2

# ------------  向量化上三角 (去对角线) ------------
vec_G1 = sym_matrix_to_vec(delta_G1, discard_diagonal=True)   # [N1 × n_edges]
vec_G2 = sym_matrix_to_vec(delta_G2, discard_diagonal=True)   # [N2 × n_edges]

# ------------  t-test (组间比较 ΔFC) ------------
t_vals, p_vals = ttest_ind(vec_G1, vec_G2,
                        nan_policy="omit")

results = pd.DataFrame(
    {"t_value": t_vals, "p_value": p_vals},
    index=[f"edge_{i}" for i in range(vec_G1.shape[1])]
)

print("Group-difference t-test on ΔFC (state 3)")
print(results.head())

# ------------ 统计显著连接数 ------------
alpha = 0.001
n_sig = (results["p_value"] < alpha).sum()
print(f"t 检验：p < {alpha} 的连边数 = {n_sig}")

In [ ]:
delta_G2.shape

In [ ]:
#############################

In [ ]:
import numpy as np
import numpy as np
from nilearn.connectome import sym_matrix_to_vec

correlation_vectors_pre = []
correlation_vectors_pre = sym_matrix_to_vec(fc_pre[1], discard_diagonal=True)

correlation_vectors_post = []
correlation_vectors_post = sym_matrix_to_vec(fc_post[1], discard_diagonal=True)

print(correlation_vectors_pre.shape)
print(correlation_vectors_post.shape)

In [ ]:
df_pre = pd.DataFrame(correlation_vectors_pre.T)
df_pre.to_excel("FCpre_cap2.xlsx", index=False, header=False) 

df_post = pd.DataFrame(correlation_vectors_post.T)
df_post.to_excel("FCpost_cap2.xlsx", index=False, header=False) 

In [ ]:
X2 = correlation_vectors_pre - correlation_vectors_post

In [ ]:
import numpy as np
np.save( "FC_CAP4_pre.npy",fc_pre[3])

In [ ]:
import numpy as np
from collections import defaultdict
from nilearn.connectome import ConnectivityMeasure

import numpy as np
from collections import defaultdict
from nilearn.connectome import ConnectivityMeasure

#  两个 ConnectivityMeasure 实例
corr_full = ConnectivityMeasure(kind='correlation',
                                vectorize=False,
                                standardize=True)      # 得到 442×442
# 不再单独建 vectorize=True 的实例，向量化自己做更快

# ② 预先算好上三角索引
n_regions  = 442
tri_idx    = np.triu_indices(n_regions, k=1)           # (2, 97 461)
vec_len    = tri_idx[0].size                           # 97 461

def compute_fc_state13_dual(source_dict,
                            state_index_dict,
                            min_tr: int = 3,
                            fill_value: float = 0.0):
    """
    拼接 state 1+3 → 返回 (fc_mats, fc_vecs)

    fc_mats : (n_subjects, 442, 442)
    fc_vecs : (n_subjects, 97 461)   （与 fc_mats 顺序一致）
    """
    subject_ids = list(source_dict.keys())
    n_subj      = len(subject_ids)

    # 初始化两个输出
    fc_mats = np.full((n_subj, n_regions, n_regions),
                      fill_value, dtype=float)
    fc_vecs = np.full((n_subj, vec_len),
                      fill_value, dtype=float)

    for subj_idx, subj_id in enumerate(subject_ids):
        # -------- 收集 state→TR --------
        per_state_ts = defaultdict(list)
        for task_name, task_data in source_dict[subj_id].items():
            if task_name not in state_index_dict[subj_id]:
                continue
            ts        = task_data['ts']                       # (T, 442)
            state_seq = state_index_dict[subj_id][task_name]  # (T,)
            for tr_row, state in zip(ts, state_seq):
                per_state_ts[state].append(tr_row)

        # -------- 拼接 state 1 + 3 --------
        tr_rows = []
        for s in (1, 3):
            tr_rows.extend(per_state_ts.get(s, []))

        if len(tr_rows) >= min_tr:
            ts_concat  = np.asarray(tr_rows)                  # (sum_TR, 442)
            fc_matrix  = corr_full.fit_transform([ts_concat])[0]
            fc_mats[subj_idx] = fc_matrix
            fc_vecs[subj_idx] = fc_matrix[tri_idx]            # 上三角展平

    return fc_mats, fc_vecs

In [ ]:
# 计算 PRE 阶段
fc_mats_pre, X_pre = compute_fc_state13_dual(
    source_dict      = cleaned_ts_dict_pre,
    state_index_dict = state_index_dict_pre,
    min_tr           = 5,      # 
    fill_value       = 0.0
)

print(fc_mats_pre.shape)   # (72, 442, 442)
print(X_pre.shape)         # (72, 97461)

In [ ]:
X_post = compute_vec_fc_state13(
    source_dict      = cleaned_ts_dict_post,        # 72 位受试者
    state_index_dict = state_index_dict_post,
    min_tr           = 5,     # 至少 5 个 TR
    fill_value       = 0.0
)

print(X_post.shape)        # -> (72, 97461)

In [ ]:
X2 = X_pre - X_post

In [ ]:
source_dict_pre.shape

In [ ]:
############################ Correlation between diff CAPs activation ############################

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
from scipy.stats import pearsonr

def compute_cap_mean_activation(state_index_dict, subject_ts_dict, cap_idx, n_regions=442):
    """
    计算某个 CAP 的平均激活（跨所有受试者、任务）
    返回 shape: (n_regions,)
    """
    ts_all = []
    for subj, task_dict in state_index_dict.items():
        for task, state_list in task_dict.items():
            ts = subject_ts_dict.get(subj, {}).get(task)
            if ts is None or len(state_list) != len(ts):
                continue
            mask = np.array(state_list) == cap_idx
            if np.sum(mask) > 0:
                ts_all.append(np.asarray(ts)[mask])
    if ts_all:
        ts_concat = np.vstack(ts_all)
        return np.mean(ts_concat, axis=0)
    else:
        return np.zeros((n_regions,))  # 如果没有该CAP的TR，则填0

# ===== 计算每个 CAP 的平均激活 =====
n_caps = 8
n_regions = 442  # 
cap_activations = []

for cap in range(n_caps):
    mean_act = compute_cap_mean_activation(
        state_index_dict=state_index_dict_post,
        subject_ts_dict=subject_ts_dict_post,
        cap_idx=cap,
        n_regions=n_regions
    )
    cap_activations.append(mean_act)

cap_activations = np.array(cap_activations)  # shape: (8, 442)

# ===== 计算两两 CAP 的相关性 =====
correlation_matrix = np.zeros((n_caps, n_caps))

for i in range(n_caps):
    for j in range(n_caps):
        r, _ = pearsonr(cap_activations[i], cap_activations[j])
        correlation_matrix[i, j] = r

# ===== 可选：保存 & 可视化 =====
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 6))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
            xticklabels=[f"CAP{i}" for i in range(n_caps)],
            yticklabels=[f"CAP{i}" for i in range(n_caps)])
plt.title("CAP Activation Correlation Matrix (Pre)", fontsize=14)
plt.tight_layout()
# plt.savefig("cap_correlation_matrix_post.svg", format="svg", dpi=300)
plt.show()

In [ ]:
############# Correlation with symptom change ------activation ##########################

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

def compute_cap_specific_activation(state_index_dict, subject_ts_dict, selected_caps):
    """
    计算指定CAP状态下的平均激活值（每个受试者，按多个任务聚合）
    """
    result = {}
    for subj, task_dict in state_index_dict.items():
        ts_all = []
        for task, state_list in task_dict.items():
            ts = subject_ts_dict.get(subj, {}).get(task)
            if ts is None:
                continue
            state_list = np.array(state_list)
            ts = np.asarray(ts)
            if len(state_list) != len(ts):
                continue  # 数据长度不一致则跳过
            mask = np.isin(state_list, selected_caps)
            if np.sum(mask) > 0:
                ts_all.append(ts[mask])
        if ts_all:
            ts_concat = np.vstack(ts_all)
            mean_activation = np.mean(ts_concat, axis=0)  # shape: (regions,)
            result[subj] = mean_activation
    return pd.DataFrame.from_dict(result, orient='index')


selected_caps = [3]

# Pre 阶段
df_cap_pre = compute_cap_specific_activation(state_index_dict_pre, subject_ts_dict_pre, selected_caps)

# Post 阶段
df_cap_post = compute_cap_specific_activation(state_index_dict_post, subject_ts_dict_post, selected_caps)

# 对齐 ID 并求差值
common_subjects = df_cap_pre.index.intersection(df_cap_post.index)
df_cap_pre = df_cap_pre.loc[common_subjects]
df_cap_post = df_cap_post.loc[common_subjects]
activation_diff = df_cap_pre.values - df_cap_post.values

# 保存
np.save("cap1_pre_post_diff.npy", activation_diff)
print(f"Saved CAP1 activation diff matrix. Shape: {activation_diff.shape}")

In [ ]:
############# Correlation with symptom change ##########################

In [ ]:
# Transform trans matrix
import numpy as np

def flatten_transition_matrix_full(matrices):
    n, k, _ = matrices.shape
    flat_matrix = matrices.reshape(n, k * k)
    return flat_matrix

In [ ]:
##
correlation_vectors_pre = []
correlation_vectors_pre = flatten_transition_matrix_full(trans_pre_matrix)  # 
print("Flattened shape:", correlation_vectors_pre.shape)  # 

correlation_vectors_post = []
correlation_vectors_post = flatten_transition_matrix_full(trans_post_matrix)  # 
print("Flattened shape:", correlation_vectors_post.shape)  # 

In [ ]:
# 计算 pre – post 差值矩阵
# significant_columns = np.arange(64)
diff_matrix = correlation_vectors_pre - correlation_vectors_post  # shape: (n_subjects, n_connections)
diff_sig = diff_matrix

# print("Difference matrix shape:", diff_matrix.shape)
print("Significant-only matrix shape:", diff_sig.shape)

In [ ]:
z_diff_sig = zscore(diff_sig, axis=0)
z_diff_dwell = zscore(Diff_dwell, axis=0)
z_diff_prop = zscore(Diff_prop_matrix, axis=0)


z_diff_sig = z_diff_sig[:, [9,13,26,27]]
z_diff_dwell = z_diff_dwell[:, [1,3]]
z_diff_prop = z_diff_prop[:, [1,3,4,6]]

# 拼接为一个新的矩阵
combined_matrix = np.hstack([z_diff_sig, z_diff_dwell, z_diff_prop])
combined_matrix.shape
## sig_diff= [9,13,27]
# sig_dwell = [1,3]
# sig_prop = [1,3,4,6]

In [ ]:
z_sig.shape

In [ ]:
# pre
df_pre = df_pre.iloc[:,1:]
df_pre = df_pre.to_numpy() 

z_sig = zscore(correlation_vectors_pre, axis=0)
z_dwell = zscore(X_dwell_pre, axis=0)
z_prop = zscore(df_pre, axis=0)

z_sig = z_sig[:, [9,13,27]]
z_dwell = z_dwell[:, [1,3]]
z_prop = z_prop[:, [1,3,4,6]]

# 拼接为一个新的矩阵
combined_matrix_pre = np.hstack([z_sig, z_dwell, z_prop])
combined_matrix_pre.shape

In [ ]:
import numpy as np
import os

save_dir = "/media/kaizhang/001/Project_2/DF" 
os.makedirs(save_dir, exist_ok=True)

np.save(os.path.join(save_dir, "combined_matrix_pre.npy"), combined_matrix_pre)

In [ ]:
#Clinical Metric
import numpy as np
import pandas as pd
# 读取数据
# 
Train_scales_file = '/media/kaizhang/001/Project_2/Covbat/CBT_only/P2_clinical_metric_remove.csv' #  P2_clinical_metric.csv P2_percentage_matrix.csv
df1 = pd.read_csv(Train_scales_file)

# clinical socres
Y = df1.iloc[:, 1:9].values 
# Y = df1.iloc[:, 1:7].values 
# Y = df1.iloc[:, [1]].values
# Y = df1.iloc[:, [2,3,5]].values

# LabelY = df1.iloc[:, 1:10].values # Arm information
Y.shape

In [ ]:
#rcca Coding
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.decomposition import PCA
import pickle
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import CCA as skCCA
from sklearn.model_selection import RepeatedKFold, KFold, LeaveOneOut, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from scipy import stats
from cca_zoo.model_selection import GridSearchCV
from cca_zoo.linear import CCA, rCCA, ElasticCCA, SCCA_IPLS

In [ ]:
#rcca Coding
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.decomposition import PCA
import pickle
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import CCA as skCCA
from sklearn.model_selection import RepeatedKFold, KFold, LeaveOneOut, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from scipy import stats
from cca_zoo.model_selection import GridSearchCV
from cca_zoo.linear import CCA, rCCA, ElasticCCA, SCCA_IPLS

def GridSearch_rCCA_bestC(X, Y, param_c, cvg):
    samp_num = X.shape[0]
    X_scores, Y_scores = np.zeros((samp_num,len(param_c))), np.zeros((samp_num,len(param_c)))
    for tr_idx, ts_idx in cvg.split(X):
        tr_X, tr_Y = X[tr_idx], Y[tr_idx]
        ts_X, ts_Y = X[ts_idx], Y[ts_idx]
        for ic,c in enumerate(param_c):
            model = rCCA(latent_dimensions=1, c=c)
            model.fit((tr_X, tr_Y))
            a, b = model.transform((ts_X, ts_Y))
            X_scores[ts_idx,ic] = a[:,0]
            Y_scores[ts_idx,ic] = b[:,0]
    rvals = np.zeros(len(param_c))
    for ic in range(len(param_c)):
        r, p = stats.pearsonr(X_scores[:,ic], Y_scores[:,ic])
        rvals[ic] = r
    # print(rvals)
    sidx = np.where(rvals==np.nanmax(rvals))[-1]
    return np.median(param_c[sidx])
        
def cross_validation_CCA(X, Y, cvg, comp_num=1, site_labels=None):
    subj_num = X.shape[0]
    
    avec = np.zeros((subj_num,comp_num))
    bvec = np.zeros((subj_num,comp_num))
    
    for tr_idx, ts_idx in cvg.split(X, site_labels):
        tr_X, tr_Y = X[tr_idx], Y[tr_idx]
        ts_X, ts_Y = X[ts_idx], Y[ts_idx]
    
        # pca = PCA(n_components=0.99)      
        # pca.fit(tr_X)
        # # print(pca.explained_variance_ratio_)
        # use_num = np.sum(pca.explained_variance_ratio_>0.01)
        # # use_num = len(pca.explained_variance_ratio_)
        # print(f'Using {use_num} PCA components')
        # tr_X = pca.transform(tr_X)[:,:use_num]
        # ts_X = pca.transform(ts_X)[:,:use_num]
     
        # pca = PCA(n_components=20)      
        # pca.fit(tr_X)
        # tr_X = pca.transform(tr_X)
        # ts_X = pca.transform(ts_X)
        
        # Y_normer = StandardScaler()
        # tr_Y = Y_normer.fit_transform(tr_Y)
        # ts_Y = Y_normer.transform(ts_Y) 
        
        # rcca
        inner_cvg = RepeatedKFold(n_splits=10, n_repeats=1, random_state=1)
        best_C = GridSearch_rCCA_bestC(tr_X, tr_Y, np.arange(0., 1, 0.05), inner_cvg)
        model = rCCA(latent_dimensions=comp_num, c=best_C)
        model.fit((tr_X, tr_Y))
        a, b = model.transform((ts_X, ts_Y))
        
        avec[ts_idx] = a
        bvec[ts_idx] = b
    return avec, bvec


# X = diff_sig
# X = Diff_prop_matrix
# X = Diff_dwell
# X = combined_matrix
X = X1
Y = Y


rep_num = 5
comp_num = 1

X = stats.zscore(X, axis=1)
site_labels = np.zeros((X.shape[0],))
# 
subj_num = X.shape[0]
print(X.shape)
print(Y.shape)
all_avec = np.zeros((rep_num,subj_num,comp_num))
all_bvec = np.zeros((rep_num,subj_num,comp_num))
for irep in range(rep_num):
    print(irep)
    # cvg = RepeatedKFold(n_splits=10, n_repeats=1, random_state=irep)
    cvg = StratifiedKFold(n_splits=10, random_state=irep+1000, shuffle=True)
    
    avec, bvec = cross_validation_CCA(X, Y, cvg, comp_num=comp_num, site_labels=site_labels)
    all_avec[irep] = avec
    all_bvec[irep] = bvec
    
real_rvals = np.nan*np.zeros((rep_num,comp_num))
for irep in range(rep_num):
    for i in range(comp_num):
        print(stats.pearsonr(all_avec[irep,:,i], all_bvec[irep,:,i]))
        r, p = stats.spearmanr(all_avec[irep,:,i], all_bvec[irep,:,i])
        real_rvals[irep,i] = r
print(np.nanmean(real_rvals, axis=0))
real_rvals

In [ ]:
################################################### Paracellation ################################

In [ ]:
import numpy as np
import nibabel as nib
from nilearn import plotting
import matplotlib.pyplot as plt
from collections import defaultdict

def plot_time_series_state_map(state_index_dict_pre_hc, subject_ts_dict, parcellation_file, label="PRE"):
    """
    - state_index_dict: dict[subject][task] = list of state labels per TR
    - subject_ts_dict: dict[subject][task] = np.array of shape (TR, region)
    - parcellation_file: nii.gz parcellation image
    - label: label for title (e.g. PRE, POST, etc.)
    """
    parcellation_img = nib.load(parcellation_file)
    parcel_map = parcellation_img.get_fdata().astype(int)
    affine = parcellation_img.affine

    n_regions = np.max(parcel_map)
    all_states = set()
    
    # collect time points for each state
    state_time_series = defaultdict(list)  # state -> list of (region,) timepoints
    
    for subj, task_dict in state_index_dict_pre_hc.items():
        for task, state_list in task_dict.items():
            ts_data = subject_ts_dict.get(subj, {}).get(task)
            if ts_data is None:
                continue
            for tr_index, state in enumerate(state_list):
                if tr_index >= len(ts_data):
                    continue
                state_time_series[state].append(ts_data[tr_index])
                all_states.add(state)

    # compute mean activation per region
    for s in sorted(all_states):
        ts = state_time_series[s]
        if len(ts) == 0:
            continue
        state_mean = np.mean(ts, axis=0)  # shape: (region,)

        # map region values to 3D volume
        region_map = np.zeros_like(parcel_map, dtype=float)
        for i in range(1, len(state_mean) + 1):
            region_map[parcel_map == i] = state_mean[i - 1]

        state_img = nib.Nifti1Image(region_map, affine)

        #plot
        plotting.plot_stat_map(state_img, title=f"State {s} - {label}",
                               display_mode='ortho', colorbar=True, vmax=0.5, vmin=-0.5, cut_coords=(0,0,0))
        plt.show()
        
# Plot
parcellation_file = "/home/kaizhang/Analysis_UIC/Brain-Mask/whole_brain_mask_Sch7net400_subcortex_cerebellum_MNI152NLin2009cAsym_res-2space.nii.gz"
plot_time_series_state_map(state_index_dict_pre, subject_ts_dict_pre, parcellation_file, label="PRE")

In [ ]:
### Save
import numpy as np
from collections import defaultdict
import pandas as pd
import os

def compute_state_centroids(state_index_dict, subject_ts_dict, k_states, zscore_within_subject=True):
    """
    计算每个state的中心点（centroid）：对属于该state的所有TR的向量求均值。
    参数
    ----
    state_index_dict: dict[subj][task] -> list[int]  # 每个TR的state标签（解码所得）
    subject_ts_dict:  dict[subj][task] -> np.ndarray of shape (T, R)  # TR x region
    k_states: int, e.g., 8
    zscore_within_subject: 是否对每个受试者的时间序列先做逐region的时序z-score（推荐）

    返回
    ----
    centroids: np.ndarray, shape (k_states, n_regions)
    counts:    np.ndarray, shape (k_states,), 每个state累计的TR数量
    """
    # 推断region维度
    # 取一个非空的 ts 矩阵来确定 n_regions
    n_regions = None
    for subj, task_dict in subject_ts_dict.items():
        for task, ts in task_dict.items():
            if ts is not None and ts.size > 0:
                n_regions = ts.shape[1]
                break
        if n_regions is not None:
            break
    if n_regions is None:
        raise ValueError("subject_ts_dict 为空或没有找到有效的 time series。")

    # 容器
    sums = np.zeros((k_states, n_regions), dtype=float)
    counts = np.zeros((k_states,), dtype=int)

    for subj, task_dict in state_index_dict.items():
        # 先对本受试者的所有任务做一次性标准化参数（按需要）
        # 这里选择“每个task单独zscore”，也可选择先拼接再zscore。
        for task, state_list in task_dict.items():
            ts = subject_ts_dict.get(subj, {}).get(task, None)
            if ts is None or len(state_list) == 0:
                continue

            # 对齐长度（稳妥处理）：若state_list比ts更长，截断；反之亦然
            T = min(len(state_list), len(ts))
            slabels = np.asarray(state_list[:T], dtype=int)
            X = ts[:T, :].astype(float)

            # 可选：对每个subject×task的time-series按region做z-score（推荐，避免幅度差异带来的偏置）
            if zscore_within_subject:
                # (T,R) -> 每列减均值除以std，std为0的列保持0
                col_mean = X.mean(axis=0, keepdims=True)
                col_std  = X.std(axis=0, ddof=1, keepdims=True)
                X = (X - col_mean) / np.where(col_std == 0, 1.0, col_std)

            # 聚合到对应state
            # 仅接受在[0, k_states-1]范围内的state标签
            valid_mask = (slabels >= 0) & (slabels < k_states)
            slabels = slabels[valid_mask]
            X = X[valid_mask]

            # 按state累加
            for s in range(k_states):
                idx = (slabels == s)
                if np.any(idx):
                    sums[s]   += X[idx].sum(axis=0)
                    counts[s] += int(idx.sum())

    # 计算均值（对count=0的state保持为0向量或抛错，按需要）
    centroids = np.zeros_like(sums)
    nonzero = counts > 0
    centroids[nonzero] = sums[nonzero] / counts[nonzero, None]

    return centroids, counts

# === 使用示例（CBT数据） ===
k_states = 8
centroids_pre, counts_pre = compute_state_centroids(state_index_dict_pre, subject_ts_dict_pre, k_states, zscore_within_subject=True)
centroids_post, counts_post = compute_state_centroids(state_index_dict_post, subject_ts_dict_post, k_states, zscore_within_subject=True)

# 保存为 .npy 和 .csv
out_dir = "./cap_centroids"
os.makedirs(out_dir, exist_ok=True)

np.save(os.path.join(out_dir, "CBT_PRE_centroids.npy"), centroids_pre)
np.save(os.path.join(out_dir, "CBT_POST_centroids.npy"), centroids_post)
np.save(os.path.join(out_dir, "CBT_PRE_counts.npy"), counts_pre)
np.save(os.path.join(out_dir, "CBT_POST_counts.npy"), counts_post)

pd.DataFrame(centroids_pre, columns=[f"R{i+1}" for i in range(centroids_pre.shape[1])]).to_csv(os.path.join(out_dir, "CBT_PRE_centroids.csv"), index_label="State")
pd.DataFrame(centroids_post, columns=[f"R{i+1}" for i in range(centroids_post.shape[1])]).to_csv(os.path.join(out_dir, "CBT_POST_centroids.csv"), index_label="State")
pd.DataFrame({"state": np.arange(k_states), "count": counts_pre}).to_csv(os.path.join(out_dir, "CBT_PRE_counts.csv"), index=False)
pd.DataFrame({"state": np.arange(k_states), "count": counts_post}).to_csv(os.path.join(out_dir, "CBT_POST_counts.csv"), index=False)

In [ ]:
out_dir

In [ ]:
import numpy as np
import nibabel as nib
from nilearn import plotting
import matplotlib.pyplot as plt
from collections import defaultdict

def plot_time_series_state_map(state_index_dict_pre_pt, subject_ts_dict, parcellation_file, label="PRE"):
    """
    - state_index_dict: dict[subject][task] = list of state labels per TR
    - subject_ts_dict: dict[subject][task] = np.array of shape (TR, region)
    - parcellation_file: nii.gz parcellation image
    - label: label for title (e.g. PRE, POST, etc.)
    """
    parcellation_img = nib.load(parcellation_file)
    parcel_map = parcellation_img.get_fdata().astype(int)
    affine = parcellation_img.affine

    n_regions = np.max(parcel_map)
    all_states = set()
    
    # collect time points for each state
    state_time_series = defaultdict(list)  # state -> list of (region,) timepoints
    
    for subj, task_dict in state_index_dict_pre_pt.items():
        for task, state_list in task_dict.items():
            ts_data = subject_ts_dict.get(subj, {}).get(task)
            if ts_data is None:
                continue
            for tr_index, state in enumerate(state_list):
                if tr_index >= len(ts_data):
                    continue
                state_time_series[state].append(ts_data[tr_index])
                all_states.add(state)

    # compute mean activation per region
    for s in sorted(all_states):
        ts = state_time_series[s]
        if len(ts) == 0:
            continue
        state_mean = np.mean(ts, axis=0)  # shape: (region,)

        # map region values to 3D volume
        region_map = np.zeros_like(parcel_map, dtype=float)
        for i in range(1, len(state_mean) + 1):
            region_map[parcel_map == i] = state_mean[i - 1]

        state_img = nib.Nifti1Image(region_map, affine)

        #plot
        plotting.plot_stat_map(state_img, title=f"State {s} - {label}",
                               display_mode='ortho', colorbar=True, vmax=0.5, vmin=-0.5, cut_coords=(0,0,0))
        plt.show()
        
# Plot
parcellation_file = "/home/kaizhang/Analysis_UIC/Brain-Mask/whole_brain_mask_Sch7net400_subcortex_cerebellum_MNI152NLin2009cAsym_res-2space.nii.gz"
plot_time_series_state_map(state_index_dict_pre_pt, subject_ts_dict_pre, parcellation_file, label="PRE")

In [ ]:
# Parecalltion Separated Plot
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from collections import defaultdict
from nilearn import datasets, surface, plotting

# 加载 fsaverage 表面（包含球面/膨胀网格等）
fsaverage = datasets.fetch_surf_fsaverage(mesh="fsaverage5")

def plot_surf_each_state_on_fsaverage(
    state_index_dict, subject_ts_dict, parcellation_file, label="PRE", hemi="right"
):
    """
    对每个 state 映射 CAP 平均值到 3D 再投影到 fsaverage 表面进行绘制。
    """
    # 加载 parcellation 图
    parcel_img = nib.load(parcellation_file)
    parcel_data = parcel_img.get_fdata().astype(int)
    affine = parcel_img.affine
    n_regions = int(np.max(parcel_data))

    state_time_series = defaultdict(list)

    # 收集各 state 的所有 TR
    for subj, task_dict in state_index_dict.items():
        for task, state_list in task_dict.items():
            ts_data = subject_ts_dict.get(subj, {}).get(task)
            if ts_data is None:
                continue
            for tr_index, state in enumerate(state_list):
                if tr_index >= len(ts_data):
                    continue
                state_time_series[state].append(ts_data[tr_index])

    # 遍历所有状态
    for s in sorted(state_time_series.keys()):
        ts = state_time_series[s]
        if len(ts) == 0:
            continue

        state_mean = np.mean(ts, axis=0)

        # 映射到体素空间
        region_map = np.zeros_like(parcel_data, dtype=float)
        for i in range(1, len(state_mean) + 1):
            region_map[parcel_data == i] = state_mean[i - 1]
        state_img = nib.Nifti1Image(region_map, affine)

        # 投影到 surface
        surf_stat_map = surface.vol_to_surf(state_img, fsaverage.pial_right if hemi == "right" else fsaverage.pial_left)

        # 绘图
        fig = plotting.plot_surf_stat_map(
            surf_mesh=fsaverage.infl_right if hemi == "right" else fsaverage.infl_left,
            stat_map=surf_stat_map,
            hemi=hemi,
            view='lateral', # lateral medial
            title=f"State {s} - {label} ({hemi})",
            colorbar=True,
            threshold=0.4,
            bg_map=fsaverage.sulc_right if hemi == "right" else fsaverage.sulc_left,
            vmax=1,
            vmin=-1
        )
        plt.show()

parcellation_file = "/home/kaizhang/Analysis_UIC/Brain-Mask/whole_brain_mask_Sch7net400_subcortex_cerebellum_MNI152NLin2009cAsym_res-2space.nii.gz"

plot_surf_each_state_on_fsaverage(
    state_index_dict=state_index_dict_post,
    subject_ts_dict=subject_ts_dict_post,
    parcellation_file=parcellation_file,
    label="Post",
    hemi="left"  # 
)

In [ ]:
## parecalltion with gradually color
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from collections import defaultdict
from nilearn import datasets, surface, plotting

fsaverage = datasets.fetch_surf_fsaverage("fsaverage5")

def plot_surf_each_state_on_fsaverage_grad(
    state_index_dict,
    subject_ts_dict,
    parcellation_file,
    label="PRE",
    hemi="right",                       # 'left' or 'right'
    views=("lateral", "medial"),
    subcere_id_start=401,
    output_dir=".",                     # ← SVG 保存目录
    min_roi_vertices=30,
    subjects=None,                      # ← 新增：仅遍历这些被试（list[str]），None=全部
    max_trs=None                        # ← 新增：每个 task 只取前 N 个 TR，None=全部
):

    # ---------- 创建输出文件夹 ----------
    os.makedirs(output_dir, exist_ok=True)

    # ---------- parcellation ----------
    parcel_img  = nib.load(parcellation_file)
    parcel_data = parcel_img.get_fdata().astype(int)
    affine      = parcel_img.affine

    subcere_mask = (parcel_data >= subcere_id_start).astype(np.uint8)
    subcere_img  = nib.Nifti1Image(subcere_mask, affine)

    # ---------- 收集时间序列 ----------
    state_ts = defaultdict(list)

    # 若指定 subjects，只遍历这些被试
    subj_items = state_index_dict.items()
    if subjects is not None:
        subj_items = ((subj, task_dict) for subj, task_dict in subj_items if subj in subjects)

    for subj, task_dict in subj_items:
        for task, state_seq in task_dict.items():
            ts = subject_ts_dict.get(subj, {}).get(task)
            if ts is None:
                continue

            # 若指定 max_trs，同步截断 state_seq 与 ts
            if max_trs is not None:
                state_seq = state_seq[:max_trs]
                ts = ts[:max_trs]

            for tr_idx, st in enumerate(state_seq):
                if tr_idx < len(ts):
                    state_ts[st].append(ts[tr_idx])

    if not state_ts:
        print("⚠︎ 未收集到任何 state 的时间序列，请检查 subjects/max_trs 是否过小、或键名是否匹配。")
        return

    # ---------- 网格 ----------
    if hemi == "right":
        infl_mesh = surface.load_surf_mesh(fsaverage.infl_right)
        pial_mesh = surface.load_surf_mesh(fsaverage.pial_right)
        sulc_bg   = fsaverage.sulc_right
    else:
        infl_mesh = surface.load_surf_mesh(fsaverage.infl_left)
        pial_mesh = surface.load_surf_mesh(fsaverage.pial_left)
        sulc_bg   = fsaverage.sulc_left

    # ---------- 逐 state ----------
    for s in sorted(state_ts.keys()):
        if not state_ts[s]:
            continue

        state_mean = np.mean(state_ts[s], axis=0)
        region_map = np.zeros_like(parcel_data, dtype=np.float32)
        for idx in range(1, len(state_mean) + 1):
            region_map[parcel_data == idx] = state_mean[idx - 1]
        state_img = nib.Nifti1Image(region_map, affine)

        # 固定颜色条范围 
        vmin, vmax = -0.8, 0.8

        # sub/cere projection
        subcere_surf = surface.vol_to_surf(subcere_img, pial_mesh)
        roi_binary   = (subcere_surf > 0)
        has_roi      = roi_binary.sum() >= min_roi_vertices

        for view in views:
            surf_stat = surface.vol_to_surf(state_img, pial_mesh)

            fig = plotting.plot_surf_stat_map(
                infl_mesh,
                stat_map=surf_stat,
                hemi=hemi,
                view=view,
                cmap="coolwarm",
                symmetric_cbar=True,
                vmin=vmin,
                vmax=vmax,
                bg_map=sulc_bg,
                title=f"State {s} – {label} ({hemi}, {view})",
                threshold=None,
                colorbar=True,
            )

            if has_roi:
                try:
                    plotting.plot_surf_contours(
                        infl_mesh,
                        roi_binary.astype(np.uint8),
                        levels=[0.5],
                        colors=["black"],
                        linewidths=1.2,
                        axes=fig.axes[0],
                    )
                except ValueError:
                    pass


            # ---------- 保存为 JPG ----------
            fname = f"State{s}_{label}_{hemi}_{view}.jpg"
            jpg_path = os.path.join(output_dir, fname)
            fig.savefig(jpg_path, format="jpg", dpi=300, bbox_inches="tight")
            plt.close(fig)              # 释放内存
            print(f"Saved: {jpg_path}")



# ===== 示例调用 =====
# 仅遍历前两个被试、每个 task 只取前 150 个 TR；若不需要，去掉 subjects/max_trs 即可。
plot_surf_each_state_on_fsaverage_grad(
    state_index_dict=state_index_dict_pre,
    subject_ts_dict=subject_ts_dict_pre,
    parcellation_file="/home/kaizhang/Analysis_UIC/Brain-Mask/whole_brain_mask_Sch7net400_subcortex_cerebellum_MNI152NLin2009cAsym_res-2space.nii.gz",
    label="Pre",
    hemi="right",
    views=("lateral", "medial"),
    output_dir="/media/kaizhang/001/Project_2/DF/FIg/FIg0919/figs_jpg",
    subjects=list(state_index_dict_pre.keys())[:2],   #限制被试
    max_trs=100                                         # 
)

In [ ]:
state_index_dict_pre

In [ ]:
def plot_surf_each_state_auto_range(
    state_index_dict,
    subject_ts_dict,
    parcellation_file,
    label="PRE",
    hemi="right",
    views=("lateral", "medial"),
    subcere_id_start=401,
    min_roi_vertices=30,
    subjects=None,
    output_dir=None   # ← 新增保存路径参数
):

    # 如果设置了保存路径，先建文件夹
    if output_dir is not None:
        os.makedirs(output_dir, exist_ok=True)

    # ---------- parcellation ----------
    parcel_img  = nib.load(parcellation_file)
    parcel_data = parcel_img.get_fdata().astype(int)
    affine      = parcel_img.affine

    subcere_mask = (parcel_data >= subcere_id_start).astype(np.uint8)
    subcere_img  = nib.Nifti1Image(subcere_mask, affine)

    # ---------- 收集时间序列 ----------
    state_ts = defaultdict(list)
    for subj, task_dict in state_index_dict.items():
        if subjects is not None and subj not in subjects:   # 只保留指定被试
            continue
        for task, state_seq in task_dict.items():
            ts = subject_ts_dict.get(subj, {}).get(task)
            if ts is None:
                continue
            for tr_idx, st in enumerate(state_seq):
                if tr_idx < len(ts):
                    state_ts[st].append(ts[tr_idx])

    # ---------- 网格 ----------
    if hemi == "right":
        infl_mesh = surface.load_surf_mesh(fsaverage.infl_right)
        pial_mesh = surface.load_surf_mesh(fsaverage.pial_right)
        sulc_bg   = fsaverage.sulc_right
    else:
        infl_mesh = surface.load_surf_mesh(fsaverage.infl_left)
        pial_mesh = surface.load_surf_mesh(fsaverage.pial_left)
        sulc_bg   = fsaverage.sulc_left

    # ---------- 逐 state ----------
    for s in sorted(state_ts.keys()):
        if not state_ts[s]:
            continue

        state_mean = np.mean(state_ts[s], axis=0)
        region_map = np.zeros_like(parcel_data, dtype=np.float32)
        for idx in range(1, len(state_mean) + 1):
            region_map[parcel_data == idx] = state_mean[idx - 1]
        state_img = nib.Nifti1Image(region_map, affine)

        # sub/cere projection
        subcere_surf = surface.vol_to_surf(subcere_img, pial_mesh)
        roi_binary   = (subcere_surf > 0)
        has_roi      = roi_binary.sum() >= min_roi_vertices

        for view in views:
            surf_stat = surface.vol_to_surf(state_img, pial_mesh)

            # 自动设定颜色范围（分位数避免极端值）
            vmin = np.nanpercentile(surf_stat, 2.5)
            vmax = np.nanpercentile(surf_stat, 97.5)
            v = max(abs(vmin), abs(vmax))
            vmin, vmax = -v, v

            fig = plotting.plot_surf_stat_map(
                infl_mesh,
                stat_map=surf_stat,
                hemi=hemi,
                view=view,
                cmap="coolwarm",
                symmetric_cbar=True,
                vmin=vmin,
                vmax=vmax,
                bg_map=sulc_bg,
                title=f"State {s} – {label} ({hemi}, {view})\n"
                      f"z∈[{surf_stat.min():.2f},{surf_stat.max():.2f}]",
                threshold=None,
                colorbar=True,
            )

            if has_roi:
                try:
                    plotting.plot_surf_contours(
                        infl_mesh,
                        roi_binary.astype(np.uint8),
                        levels=[0.5],
                        colors=["black"],
                        linewidths=1.2,
                        axes=fig.axes[0],
                    )
                except ValueError:
                    pass

            # 展示
            plt.show()

            # 保存
            if output_dir is not None:
                fname = f"State{s}_{label}_{hemi}_{view}.png"
                fpath = os.path.join(output_dir, fname)
                fig.savefig(fpath, dpi=300, bbox_inches="tight")
                print(f"已保存: {fpath}")
            plt.close(fig)

In [ ]:
fpath

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from collections import defaultdict
from nilearn import datasets, surface, plotting
import scipy.stats as stats

def compare_state_activation_on_surface_cap(
    state_index_dict_pre,
    state_index_dict_post,
    subject_ts_pre,            # dict: subj->(T, P) 或 list[(T,P), ...]
    subject_ts_post,           # 同上
    parcellation_file,         # 体素级标签图（24 ROI；标签值未必是1..24）
    hemi="left",               # "left" 或 "right"
    threshold=0.0,             # 可视化阈值（t值）
    vmax=5.0, vmin=-5.0,       # 颜色范围
    mesh="fsaverage5"
):
    """
    对每个 CAP state，收集 pre / post 对应帧的 ROI 激活（24列），
    做 Welch t-test (post vs pre)，生成每个 state 的 t-map，并投影到皮层表面。
    兼容：
      - state_index_dict_* : {subj: [state,...]} 或 {subj: {task: [state,...]}}
      - subject_ts_*       : {subj: (T,P)} 或 list[(T,P), ...]
    """

    # ---- 1) 取 fsaverage 网格 ----
    fsavg = datasets.fetch_surf_fsaverage(mesh=mesh)

    # ---- 2) 读取分区体积并准备标签映射 ----
    parcel_img = nib.load(parcellation_file)
    parcel_data = parcel_img.get_fdata().astype(int)
    affine = parcel_img.affine

    # 实际存在的正标
    uniq_labels = np.array(sorted([v for v in np.unique(parcel_data) if v > 0]))
    P = uniq_labels.size  # 期望列数（一般=24）

    # ---- 3) 把 subject 时序转成 dict（若传入的是 list）----
    def to_dict_ts(obj):
        if isinstance(obj, dict):
            return obj
        elif isinstance(obj, list):
            return {i: np.asarray(ts) for i, ts in enumerate(obj)}
        else:
            raise TypeError(f"subject_ts_* 必须是 dict 或 list，当前是 {type(obj)}")
    subject_ts_pre = to_dict_ts(subject_ts_pre)
    subject_ts_post = to_dict_ts(subject_ts_post)

    # ---- 4) 兼容 state_index 单层/双层：统一生成 subj->list[int] 的状态序列 ----
    def extract_state_seq_per_subj(state_dict):
        out = {}
        for subj, val in state_dict.items():
            if isinstance(val, dict):
                # 多任务时拼接所有任务的序列（时间顺序未知则按字典顺序，实际最好保存顺序）
                seq_all = []
                for seq in val.values():
                    if seq is None: 
                        continue
                    seq_all.extend(list(np.asarray(seq, dtype=int).ravel()))
                out[subj] = seq_all
            elif isinstance(val, (list, np.ndarray)):
                out[subj] = list(np.asarray(val, dtype=int).ravel())
            else:
                raise TypeError(f"state_index[{subj}] 类型不支持: {type(val)}")
        return out

    state_seq_pre  = extract_state_seq_per_subj(state_index_dict_pre)
    state_seq_post = extract_state_seq_per_subj(state_index_dict_post)

    # ---- 5) 收集每个 state 的 pre/post 帧（按 ROI 列）----
    # state_ts_pre[s] 会堆叠多被试的帧（N_s_pre, P）
    state_ts_pre  = defaultdict(list)
    state_ts_post = defaultdict(list)

    # 遍历所有被试（取 pre/post 的交集）
    common_subj = sorted(set(subject_ts_pre.keys()) & set(subject_ts_post.keys()) & set(state_seq_pre.keys()) & set(state_seq_post.keys()))
    if len(common_subj) == 0:
        raise RuntimeError("没有共同的被试（pre/post/state）可用。")

    for subj in common_subj:
        ts_pre  = np.asarray(subject_ts_pre[subj],  dtype=float)
        ts_post = np.asarray(subject_ts_post[subj], dtype=float)
        states_pre  = np.asarray(state_seq_pre[subj],  dtype=int)
        states_post = np.asarray(state_seq_post[subj], dtype=int)

        # 基本检查与对齐
        if ts_pre.ndim != 2 or ts_post.ndim != 2:
            continue
        if ts_pre.shape[1] != P or ts_post.shape[1] != P:
            print(f"[Warn] subj {subj}: 列数与分区数不一致 (pre {ts_pre.shape}, post {ts_post.shape}), 期望列数={P}，跳过该被试。")
            continue

        T_pre  = ts_pre.shape[0]
        T_post = ts_post.shape[0]
        if len(states_pre) != T_pre:
            m = min(len(states_pre), T_pre)
            if m == 0: 
                continue
            print(f"[Warn] subj {subj}: pre 状态序列与帧数不等，按最小长度截断到 {m}")
            states_pre = states_pre[:m]
            ts_pre     = ts_pre[:m]
        if len(states_post) != T_post:
            m = min(len(states_post), T_post)
            if m == 0: 
                continue
            print(f"[Warn] subj {subj}: post 状态序列与帧数不等，按最小长度截断到 {m}")
            states_post = states_post[:m]
            ts_post     = ts_post[:m]

        # 把该被试的帧按 state 分桶
        for s in np.unique(states_pre):
            state_ts_pre[s].append(ts_pre[states_pre == s])
        for s in np.unique(states_post):
            state_ts_post[s].append(ts_post[states_post == s])

    # 把分桶后的 list[list[frames]] → 堆叠为 ndarray
    for s in list(state_ts_pre.keys()):
        if len(state_ts_pre[s]) > 0:
            state_ts_pre[s] = np.vstack(state_ts_pre[s])   # (N_s_pre,  P)
    for s in list(state_ts_post.keys()):
        if len(state_ts_post[s]) > 0:
            state_ts_post[s] = np.vstack(state_ts_post[s]) # (N_s_post, P)

    # ---- 6) 遍历 state 做 Welch t-test，并绘图 ----
    states_all = sorted(set(state_ts_pre.keys()) & set(state_ts_post.keys()))
    if len(states_all) == 0:
        print("没有共同的 state 可比较。")
        return

    for s in states_all:
        pre_data  = state_ts_pre[s]   # (N_pre,  P)
        post_data = state_ts_post[s]  # (N_post, P)

        if not isinstance(pre_data, np.ndarray) or not isinstance(post_data, np.ndarray):
            print(f"State {s} 数据不足，跳过。")
            continue
        if pre_data.shape[0] < 5 or post_data.shape[0] < 5:
            print(f"State {s} 数据不足（pre={pre_data.shape[0]}, post={post_data.shape[0]}），跳过。")
            continue

        # Welch t-test：post vs pre（逐 ROI）
        t_vals, p_vals = stats.ttest_ind(post_data, pre_data, axis=0, equal_var=False, nan_policy='omit')
        t_vals = np.nan_to_num(t_vals, nan=0.0, posinf=0.0, neginf=0.0)

        if t_vals.size != P:
            print(f"[Warn] State {s}: t_vals长度={t_vals.size} 与分区数 P={P} 不一致；按最小长度映射")
        L = min(t_vals.size, P)

        # 把 ROI 向量回填到 3D 体素（按 uniq_labels 顺序）
        region_map = np.zeros_like(parcel_data, dtype=float)
        for i, lab in enumerate(uniq_labels[:L]):
            region_map[parcel_data == lab] = t_vals[i]
        t_img = nib.Nifti1Image(region_map, affine)

        # 体素 -> 表面（选择半球）
        if hemi == "right":
            surf_mesh = fsavg.infl_right
            pial = fsavg.pial_right
            sulc = fsavg.sulc_right
        else:
            surf_mesh = fsavg.infl_left
            pial = fsavg.pial_left
            sulc = fsavg.sulc_left

        surf_stat_map = surface.vol_to_surf(t_img, pial)

        # 可视化
        plotting.plot_surf_stat_map(
            surf_mesh=fsavg.infl_left,
            stat_map=surf_stat_map,
            hemi="left",
            view="lateral",
            title=f"State {s} Post - Pre (t-map, left)",
            colorbar=True,
            threshold=2.0,            # 
            cmap="RdBu_r",            # 
            bg_map=fsavg.sulc_left,
            bg_on_data=True,          # 
            darkness=0.4,             # 调整背景亮度
            vmax=6,
            vmin=-6
        )


# ==== 调用（与你当前变量名对应）====
compare_state_activation_on_surface_cap(
    state_index_dict_pre=state_index_dict_pre,
    state_index_dict_post=state_index_dict_post,
    subject_ts_pre=subject_ts_dict_pre,     # 
    subject_ts_post=subject_ts_dict_post,   # 
    parcellation_file="/home/kaizhang/Analysis_UIC/Brain-Mask/extendNet_24regions_updateAmy_MNI152NLin2009cAsym_res-2_space.nii.gz",
    hemi="left",
    threshold=1.96,   # 
    vmax=6, vmin=-6
)

In [ ]:
# Comparation between post-pre with gradually color (plot single state) with FDR
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from collections import defaultdict

from nilearn import datasets, surface, plotting
import scipy.stats as stats
from statsmodels.stats.multitest import fdrcorrection


# ------------------------------------------------------------------
#  Main function
# ------------------------------------------------------------------
def compare_state_activation_on_surface_grad(
    state_index_dict_pre,
    state_index_dict_post,
    subject_ts_dict_pre,
    subject_ts_dict_post,
    parcellation_file,
    hemi="right",
    views=("lateral", "medial"),
    vmin=-10,
    vmax=10,
    subcere_id_start=401,
    min_roi_vertices=30,
    fdr_alpha=0.05,
    states_to_plot={1},          #  state
    min_common_subjs=1,             # 
    output_dir=".",
):
    os.makedirs(output_dir, exist_ok=True)

    # ----------- template & parcellation -------------------------
    fsaverage   = datasets.fetch_surf_fsaverage("fsaverage5")
    parcel_img  = nib.load(parcellation_file)
    parcel_data = parcel_img.get_fdata().astype(int)
    affine      = parcel_img.affine

    subcere_mask = (parcel_data >= subcere_id_start).astype(np.uint8)
    subcere_img  = nib.Nifti1Image(subcere_mask, affine)

    # ----------- surface meshes & background ---------------------
    if hemi == "right":
        infl_mesh = surface.load_surf_mesh(fsaverage.infl_right)
        pial_mesh = surface.load_surf_mesh(fsaverage.pial_right)
        sulc_bg   = fsaverage.sulc_right
    else:
        infl_mesh = surface.load_surf_mesh(fsaverage.infl_left)
        pial_mesh = surface.load_surf_mesh(fsaverage.pial_left)
        sulc_bg   = fsaverage.sulc_left

    subcere_surf = surface.vol_to_surf(subcere_img, pial_mesh)
    roi_binary   = (subcere_surf > 0)
    has_roi      = roi_binary.sum() >= min_roi_vertices

    # ----------- collect TR → subject-level dict -----------------
    subj_state_pre  = defaultdict(lambda: defaultdict(list))
    subj_state_post = defaultdict(lambda: defaultdict(list))

    # -- pre
    for subj, task_dict in state_index_dict_pre.items():
        for task, states in task_dict.items():
            ts = subject_ts_dict_pre.get(subj, {}).get(task)          # (nTR, 442)
            if ts is None:
                continue
            for i, st in enumerate(states):
                if i < len(ts):
                    subj_state_pre[st][subj].append(ts[i])

    # -- post
    for subj, task_dict in state_index_dict_post.items():
        for task, states in task_dict.items():
            ts = subject_ts_dict_post.get(subj, {}).get(task)
            if ts is None:
                continue
            for i, st in enumerate(states):
                if i < len(ts):
                    subj_state_post[st][subj].append(ts[i])

    # ----------- diagnostics ------------------------------------
    print(">>> 仅绘制 State:", sorted(states_to_plot))
    print(">>> 每个 state 共同受试者数：")
    for s in sorted(states_to_plot):
        n_common = len(set(subj_state_pre[s]) & set(subj_state_post[s]))
        print(f"    State {s}: {n_common}")
    print()

    # ----------- iterate over selected states --------------------
    for s in sorted(states_to_plot):
        if s not in subj_state_pre or s not in subj_state_post:
            print(f"State {s}: pre 或 post 数据缺失，跳过。")
            continue

        common_subjs = sorted(set(subj_state_pre[s]) & set(subj_state_post[s]))
        if len(common_subjs) < min_common_subjs:
            print(f"State {s}: 共同受试者不足（{len(common_subjs)}），跳过。")
            continue

        # ---- subject-level mean → (Nsubj, 442) ------------------
        pre  = np.stack([
            np.mean(subj_state_pre[s][subj], axis=0, dtype=np.float32)
            for subj in common_subjs
        ])
        post = np.stack([
            np.mean(subj_state_post[s][subj], axis=0, dtype=np.float32)
            for subj in common_subjs
        ])

         # ---- paired t-test -------------------------------------
        t_vals, p_vals = stats.ttest_rel(post, pre, axis=0, nan_policy="omit")
        t_vals = np.nan_to_num(t_vals)
        p_vals = np.nan_to_num(p_vals, nan=1.0)

        #Z
        z_vals  = stats.norm.isf(p_vals / 2.0) * np.sign(t_vals)

        
        # Debug  
        print(f"[State {s}]  "
          f"t-val range: {t_vals.min():.2f} ~ {t_vals.max():.2f}   "
          f"z-val range: {z_vals.min():.2f} ~ {z_vals.max():.2f}   "
          f"p-val min: {p_vals.min():.3g}")
        
        # ---- FDR ----        
        # # # t value
        # reject = p_vals < 0.05
        # t_vals_fdr = np.where(reject, t_vals, 0)
        
        # # z value
        # reject = p_vals < 0.05
        # z_vals_masked = np.where(reject, z_vals, 0.0)  

        #z value without threshold
        z_vals_masked = z_vals 
        
        # reject, _  = fdrcorrection(p_vals, alpha=fdr_alpha, method="indep")
        # print(f"[State {s}]  #ROI passing FDR={fdr_alpha}: {reject.sum()} / {len(reject)}")
        # z_vals_masked = np.where(reject, z_vals, 0.0)  
        
        # # ---- FDR correction ------------------------------------
        # reject, _  = fdrcorrection(p_vals, alpha=fdr_alpha, method="indep")
        # t_vals_fdr = np.where(reject, t_vals, 0)                 # non-sig → 0

        # # ---- ROI → 3-D volume ----------------------------------
        # #t value
        # region_map = np.zeros_like(parcel_data, dtype=np.float32)
        # for idx in range(1, len(t_vals_fdr) + 1):
        #     region_map[parcel_data == idx] = t_vals_fdr[idx - 1]
        # t_img = nib.Nifti1Image(region_map, affine)

        # z value
        region_map = np.zeros_like(parcel_data, dtype=np.float32)
        for idx in range(1, len(z_vals_masked) + 1):
            region_map[parcel_data == idx] = z_vals_masked[idx - 1]
        t_img = nib.Nifti1Image(region_map, affine)  

        # ---- plot & save ---------------------------------------
        for view in views:
            surf_stat = surface.vol_to_surf(t_img, pial_mesh)

            fig = plotting.plot_surf_stat_map(
                infl_mesh,
                stat_map      = surf_stat,
                hemi          = hemi,
                view          = view,
                cmap          = "coolwarm",
                symmetric_cbar=True,
                vmin          = vmin,
                vmax          = vmax,
                bg_map        = sulc_bg,
                title         = f"State {s}  Post – Pre (FDR q<{fdr_alpha})",
                threshold     = None,
                colorbar      = True,
            )

            if has_roi:
                try:
                    plotting.plot_surf_contours(
                        infl_mesh,
                        roi_binary.astype(np.uint8),
                        levels     =[0.5],
                        colors     =["black"],
                        linewidths =1.2,
                        axes       =fig.axes[0],
                    )
                except ValueError:
                    pass

            fname = f"State{s}_Tmap_FDR_{hemi}_{view}.svg"
            fig.savefig(os.path.join(output_dir, fname), format="svg", dpi=300)
            plt.close(fig)
            print(f"Saved: {os.path.join(output_dir, fname)}")


# ------------------------------------------------------------------
#  Example call
# ------------------------------------------------------------------
if __name__ == "__main__":
    compare_state_activation_on_surface_grad(
        state_index_dict_pre   = state_index_dict_pre,
        state_index_dict_post  = state_index_dict_post,
        subject_ts_dict_pre    = subject_ts_dict_pre,
        subject_ts_dict_post   = subject_ts_dict_post,
        parcellation_file      = (
            "/home/kaizhang/Analysis_UIC/Brain-Mask/"
            "whole_brain_mask_Sch7net400_subcortex_cerebellum_MNI152NLin2009cAsym_res-2space.nii.gz"
        ),
        hemi                   = "left",
        views                  = ("lateral", "medial"),
        vmin                   = -3,
        vmax                   = 3,
        # fdr_alpha              = 0.05,
        states_to_plot         = {1},      
        min_common_subjs       = 1,         
        output_dir             = "/media/kaizhang/001/Project_2/DF/FIg/0527/Tmaps_svg",
    )

In [ ]:
# ------------------------------------------------------------------
#  MODULE A:  build_and_save_state_matrices.py
# ------------------------------------------------------------------
import os
import numpy as np
from collections import defaultdict

def build_and_save_state_matrices(
    state_index_dict_pre,
    state_index_dict_post,
    subject_ts_dict_pre,
    subject_ts_dict_post,
    states_to_process={1},
    min_common_subjs=1,
    out_dir="state_matrices"
):
    os.makedirs(out_dir, exist_ok=True)

    # ---------- 把每个 state → {subj: [TR向量]} ----------
    subj_state_pre  = defaultdict(lambda: defaultdict(list))
    subj_state_post = defaultdict(lambda: defaultdict(list))

    for subj, task_dict in state_index_dict_pre.items():
        for task, states in task_dict.items():
            ts = subject_ts_dict_pre.get(subj, {}).get(task)  # (nTR, 442)
            if ts is None:
                continue
            for i, st in enumerate(states):
                if i < len(ts):
                    subj_state_pre[st][subj].append(ts[i])

    for subj, task_dict in state_index_dict_post.items():
        for task, states in task_dict.items():
            ts = subject_ts_dict_post.get(subj, {}).get(task)
            if ts is None:
                continue
            for i, st in enumerate(states):
                if i < len(ts):
                    subj_state_post[st][subj].append(ts[i])

    # ---------- 计算每个 state 的 Pre / Post / Diff 并保存 ----------
    for s in sorted(states_to_process):
        common = sorted(set(subj_state_pre[s]) & set(subj_state_post[s]))
        if len(common) < min_common_subjs:
            print(f"State {s}: 共同受试者不足（{len(common)}），跳过。")
            continue

        pre_mat  = np.stack([np.mean(subj_state_pre[s][subj],  axis=0, dtype=np.float32)
                             for subj in common])          # (Nsubj, 442)
        post_mat = np.stack([np.mean(subj_state_post[s][subj], axis=0, dtype=np.float32)
                             for subj in common])          # (Nsubj, 442)
        diff_mat = post_mat - pre_mat                      # (Nsubj, 442)

        # CovBat / 其他批次方法通常要求 (features, samples)，因此转置再保存
        np.save(os.path.join(out_dir, f"State{s}_pre.npy"),   pre_mat.T)
        np.save(os.path.join(out_dir, f"State{s}_post.npy"),  post_mat.T)
        np.save(os.path.join(out_dir, f"State{s}_diff.npy"),  diff_mat.T)

        print(f"State {s}  已保存：Pre / Post / Diff  →  {out_dir}")

# ------------------------ 示例调用 ------------------------
if __name__ == "__main__":
    build_and_save_state_matrices(
        state_index_dict_pre,   # 您已有的字典
        state_index_dict_post,
        subject_ts_dict_pre,
        subject_ts_dict_post,
        states_to_process={1},     # 需要处理的 state
        min_common_subjs=2,
        out_dir="/media/kaizhang/001/state_matrices"
    )

In [ ]:
# ------------------------------------------------------------------
#  MODULE A (xlsx version):  build_and_save_state_matrices.py
# ------------------------------------------------------------------
import os
import numpy as np
import pandas as pd
from collections import defaultdict

def build_and_save_state_matrices(
    state_index_dict_pre,
    state_index_dict_post,
    subject_ts_dict_pre,
    subject_ts_dict_post,
    states_to_process={1},
    min_common_subjs=1,
    out_dir="state_matrices"
):
    os.makedirs(out_dir, exist_ok=True)

    # ---------- 整理 TR → subject-level dict ----------
    subj_state_pre  = defaultdict(lambda: defaultdict(list))
    subj_state_post = defaultdict(lambda: defaultdict(list))

    for subj, task_dict in state_index_dict_pre.items():
        for task, states in task_dict.items():
            ts = subject_ts_dict_pre.get(subj, {}).get(task)
            if ts is None:
                continue
            for i, st in enumerate(states):
                if i < len(ts):
                    subj_state_pre[st][subj].append(ts[i])

    for subj, task_dict in state_index_dict_post.items():
        for task, states in task_dict.items():
            ts = subject_ts_dict_post.get(subj, {}).get(task)
            if ts is None:
                continue
            for i, st in enumerate(states):
                if i < len(ts):
                    subj_state_post[st][subj].append(ts[i])

    # ---------- 计算并保存 ----------
    for s in sorted(states_to_process):
        common = sorted(set(subj_state_pre[s]) & set(subj_state_post[s]))
        if len(common) < min_common_subjs:
            print(f"State {s}: 共同受试者不足（{len(common)}），跳过。")
            continue

        pre_mat  = np.stack([np.mean(subj_state_pre[s][subj],  axis=0, dtype=np.float32)
                             for subj in common])            # (Nsubj, 442)
        post_mat = np.stack([np.mean(subj_state_post[s][subj], axis=0, dtype=np.float32)
                             for subj in common])            # (Nsubj, 442)
        diff_mat = post_mat - pre_mat                        # (Nsubj, 442)

        # —— 保存为 .npy（features, samples）——
        np.save(os.path.join(out_dir, f"State{s}_pre.npy"),   pre_mat.T)
        np.save(os.path.join(out_dir, f"State{s}_post.npy"),  post_mat.T)
        np.save(os.path.join(out_dir, f"State{s}_diff.npy"),  diff_mat.T)

        # —— 同步写成 .xlsx（samples, features），便于查看 —— 
        pd.DataFrame(pre_mat ).to_excel(f"{out_dir}/State{s}_pre.xlsx",
                                        index=False, header=False)
        pd.DataFrame(post_mat).to_excel(f"{out_dir}/State{s}_post.xlsx",
                                        index=False, header=False)
        pd.DataFrame(diff_mat).to_excel(f"{out_dir}/State{s}_diff.xlsx",
                                        index=False, header=False)

        print(f"State {s}  已保存为 .npy 及 .xlsx →  {out_dir}")

# ---------------------- 示例调用 --------------------------
if __name__ == "__main__":
    build_and_save_state_matrices(
        state_index_dict_pre,
        state_index_dict_post,
        subject_ts_dict_pre,
        subject_ts_dict_post,
        states_to_process={1, 3},
        min_common_subjs=2,
        out_dir="/media/kaizhang/001/state_matrices"
    )

In [ ]:
# ------------------------------------------------------------
# 统计哪些 state 在 Pre→Post 发生了更显著的变化
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multitest import fdrcorrection
import matplotlib.pyplot as plt
from collections import defaultdict

def summarize_state_changes(
    state_index_dict_pre,
    state_index_dict_post,
    subject_ts_dict_pre,
    subject_ts_dict_post,
    alpha=0.05,          # FDR 阈值
    min_samples=5        # 至少多少 TR 才分析
):
    """
    返回 DataFrame，包含：
        state, n_sig, sum_abs_t, mean_abs_t
    """
    # ---------- 收集每个 state 的 TR ----------
    st_pre, st_post = defaultdict(list), defaultdict(list)

    for subj, task_dict in state_index_dict_pre.items():
        for task, states in task_dict.items():
            ts = subject_ts_dict_pre.get(subj, {}).get(task)
            if ts is None: continue
            for i, st in enumerate(states):
                if i < len(ts): st_pre[st].append(ts[i])

    for subj, task_dict in state_index_dict_post.items():
        for task, states in task_dict.items():
            ts = subject_ts_dict_post.get(subj, {}).get(task)
            if ts is None: continue
            for i, st in enumerate(states):
                if i < len(ts): st_post[st].append(ts[i])

    # ---------- 计算指标 ----------
    summary = []
    for s in sorted(set(st_pre) & set(st_post)):
        pre  = np.asarray(st_pre[s])
        post = np.asarray(st_post[s])
        if pre.shape[0] < min_samples or post.shape[0] < min_samples:
            continue

        t_vals, p_vals = stats.ttest_ind(
            post, pre, axis=0, equal_var=False, nan_policy="omit"
        )
        t_vals = np.nan_to_num(t_vals)
        p_vals = np.nan_to_num(p_vals, nan=1.0)

        reject, _ = fdrcorrection(p_vals, alpha=alpha, method="indep")

        n_sig      = int(reject.sum())
        abs_t_sig  = np.abs(t_vals[reject])
        sum_abs_t  = float(abs_t_sig.sum())
        mean_abs_t = float(abs_t_sig.mean()) if n_sig else 0.0

        summary.append({
            "state": s,
            "n_sig": n_sig,
            "sum_abs_t": sum_abs_t,
            "mean_abs_t": mean_abs_t,
        })

    df = (pd.DataFrame(summary)
            .sort_values("n_sig", ascending=False)
            .reset_index(drop=True))
    return df

# ---------------- 运行并查看 ----------------
df_summary = summarize_state_changes(
    state_index_dict_pre,
    state_index_dict_post,
    subject_ts_dict_pre,
    subject_ts_dict_post,
    alpha=0.05
)

print("\n=== Pre→Post 变化量化结果 ===")
print(df_summary)

# ---------------- 可选：柱状图 ----------------
plt.figure(figsize=(6,4))
plt.bar(df_summary["state"], df_summary["n_sig"], color="black")
plt.xlabel("State")
plt.ylabel("# Significant parcels (FDR q<0.05)")
plt.title("Pre→Post change per state")
plt.tight_layout()
plt.show()

In [ ]:
## Bold signal plot

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def plot_bold_timeseries_minimal(
    bold_ts,
    region_indices=None,
    tr=2.0,
    subject_id="sub-001",
    task_name="TASK",
    zscore=False
):
    """
    绘制简洁版 BOLD 信号图（无 grid，无 legend，无 vertical bars）
    """
    if region_indices is None:
        region_indices = list(range(5))

    bold_selected = bold_ts[:, region_indices]

    if zscore:
        from scipy.stats import zscore
        bold_selected = zscore(bold_selected, axis=0)

    time_axis = np.arange(bold_selected.shape[0]) * tr

    sns.set_style("white")  # 无背景网格
    plt.figure(figsize=(12, 4))

    # 绘制 BOLD 曲线（不显示 legend）
    for i in range(bold_selected.shape[1]):
        plt.plot(time_axis, bold_selected[:, i], linewidth=1.5)

    plt.xlabel("Time (s)", fontsize=12)
    plt.ylabel("BOLD Signal", fontsize=12)
    plt.title(f"{subject_id} - {task_name}", fontsize=14)

    sns.despine()
    plt.tight_layout()
    plt.show()


bold_ts = subject_ts_dict_pre["sub-PCS006"]["REST"]
# 绘图
# plot_bold_timeseries(bold_ts, region_indices=[10, 22, 55, 100, 180], tr=2.0, subject_id="subj_001", task_name="REST")

plot_bold_timeseries_minimal(
    bold_ts=bold_ts,
    region_indices=[10, 22, 55,100],
    tr=2.0,
    subject_id="subj_001",
    task_name="REST"
)



In [ ]:
################## 雷达图  ####################

In [ ]:
import pandas as pd
import numpy as np

# 读取标签文件
csv_path = "/home/kaizhang/Analysis_UIC/Brain-Mask/whole_brain_mask_Sch7net400_subcortex_cerebellum1.csv" # whole_brain_mask_Sch7net400_subcortex_cerebellum1  whole_brain_mask_Sch7net400_2
df = pd.read_csv(csv_path)

# 提取网络名，并编码成整数 
unique_networks = sorted(df['network_name'].unique())  # 保证顺序一致
network_name_to_index = {name: idx for idx, name in enumerate(unique_networks)}

network_names = unique_networks          # 
# 创建网络索引数组
network_labels = df['network_name'].map(network_name_to_index).to_numpy()  # 

In [ ]:
network_names

In [ ]:
network_labels

In [ ]:
# 400
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

def plot_radar_each_state_pos_neg(
    state_index_dict,
    subject_ts_dict,
    network_labels,          # 长度 = 400，对应皮层 parcel
    network_names,           # 7 个网络名 (或你的自定义顺序)
    label="Post"
):
    """
    为每个 brain state 画雷达图：红 = 正激活，蓝 = 负激活。
    仅使用 parcellation 前 400 个皮层 parcel；忽略小脑/皮下 42 个 parcel。
    """
    n_net   = len(network_names)
    theta   = np.linspace(0, 2*np.pi, n_net, endpoint=False)
    angles  = np.concatenate([theta, theta[:1]])   # 闭环

    # ---------- 收集各 state 的 TR ----------
    state_ts = defaultdict(list)
    for subj, task_dict in state_index_dict.items():
        for task, state_seq in task_dict.items():
            ts = subject_ts_dict.get(subj, {}).get(task)      # ts.shape = (TR, 442)
            if ts is None:
                continue
            for tr_idx, state_id in enumerate(state_seq):
                if tr_idx < len(ts):
                    state_ts[state_id].append(ts[tr_idx])

    # ---------- 逐 state 绘图 ----------
    for s in sorted(state_ts.keys()):
        if not state_ts[s]:
            continue

        # ① region → network 聚合（仅前 400 个皮层 parcel）
        state_mean_full   = np.mean(state_ts[s], axis=0)      # shape=(442,)
        state_mean_cortex = state_mean_full[:len(network_labels)]  # (400,)

        net_means = np.array([
            state_mean_cortex[network_labels == k].mean()
            for k in range(n_net)
        ])

        # ② 正负分离
        pos_vals = np.clip(net_means, 0, None)                # ≥0
        neg_vals = -np.clip(net_means, None, 0)               # <0 取绝对值

        pos_plot = np.concatenate([pos_vals, pos_vals[:1]])
        neg_plot = np.concatenate([neg_vals, neg_vals[:1]])

        r_max = max(pos_plot.max(), neg_plot.max(), 1e-6) * 1.05

        # ③ 画图
        fig, ax = plt.subplots(figsize=(5, 5), subplot_kw=dict(polar=True))

        # 灰色虚线网格
        ax.spines["polar"].set_visible(False)  
        ax.set_ylim(0, r_max)
        ax.yaxis.grid(True, linestyle="--", color="grey", linewidth=0.6)
        ax.xaxis.grid(False)

        # 黑色径向轴
        for t in theta:
            ax.plot([t, t], [0, r_max], color="black", linewidth=3)

        # 数据曲线
        ax.plot(angles, pos_plot, color="red",  linewidth=3, label="Positive network")
        ax.plot(angles, neg_plot, color="blue", linewidth=3, label="Negative network")

        # 可选填充：若想仅线条，可注释掉下一行
        # ax.fill(angles, pos_plot, color="red",  alpha=0.15)
        # ax.fill(angles, neg_plot, color="blue", alpha=0.15)

        # 轴标签与标题
        ax.set_xticks(theta)
        ax.set_xticklabels(network_names, fontsize=9)
        ax.set_yticklabels([])        # 隐藏径向刻度数字
        # ax.set_title(f"State {s} – {label}", fontsize=14, pad=20)
        ax.legend(loc="upper right", bbox_to_anchor=(1.15, 1.1), frameon=False)

        plt.tight_layout()
        plt.show()
        
plot_radar_each_state_pos_neg(
    state_index_dict=state_index_dict_post,
    subject_ts_dict=subject_ts_dict_post,
    network_labels=network_labels,   # 长度=400
    network_names=network_names,     # ["CN", "SCN", "VN", "SMN", "DMN", "ATN", ...]
    label="Post"
)


In [ ]:
# 雷达图 400 Save
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict


def plot_radar_each_state_pos_neg(
    state_index_dict,
    subject_ts_dict,
    network_labels,          # 长度 = 400，对应皮层 parcel
    network_names,           # 虽然不再显示，但仍需用来计数
    label="Post",
    output_dir="."           # 保存 SVG 的文件夹
):
    os.makedirs(output_dir, exist_ok=True)

    n_net   = len(network_names)
    theta   = np.linspace(0, 2*np.pi, n_net, endpoint=False)
    angles  = np.concatenate([theta, theta[:1]])

    # ---------- 收集每个 state 的 TR ----------
    state_ts = defaultdict(list)
    for subj, task_dict in state_index_dict.items():
        for task, state_seq in task_dict.items():
            ts = subject_ts_dict.get(subj, {}).get(task)
            if ts is None:
                continue
            for tr_idx, st in enumerate(state_seq):
                if tr_idx < len(ts):
                    state_ts[st].append(ts[tr_idx])

    # ---------- 逐 state 绘图 ----------
    for s in sorted(state_ts.keys()):
        if not state_ts[s]:
            continue

        # ① region → network 聚合（仅前 400 皮层 parcel）
        state_mean_full   = np.mean(state_ts[s], axis=0)          # (442,)
        state_mean        = state_mean_full[:len(network_labels)] # (400,)

        net_means = np.array([
            state_mean[network_labels == k].mean()
            for k in range(n_net)
        ])

        pos_vals = np.clip(net_means, 0, None)
        neg_vals = -np.clip(net_means, None, 0)

        pos_plot = np.concatenate([pos_vals, pos_vals[:1]])
        neg_plot = np.concatenate([neg_vals, neg_vals[:1]])

        r_max = max(pos_plot.max(), neg_plot.max(), 1e-6) * 1.05

        # ② 画图
        fig, ax = plt.subplots(figsize=(5, 5), subplot_kw=dict(polar=True))
        
        ax.spines["polar"].set_visible(False)  
        ax.set_ylim(0, r_max)
        ax.yaxis.grid(True, linestyle="--", color="grey", linewidth=0.6)
        ax.xaxis.grid(False)

        for t in theta:
            ax.plot([t, t], [0, r_max], color="black", linewidth=3)

        ax.plot(angles, pos_plot, color="red",  linewidth=3, label="Positive")
        ax.plot(angles, neg_plot, color="blue", linewidth=3, label="Negative")

        # ★ 去掉网络名称标签 ★
        ax.set_xticks(theta)
        ax.set_xticklabels([])          
        ax.set_yticklabels([])

        # ax.legend(loc="upper right", bbox_to_anchor=(1.1, 1.05), frameon=False)

        # ③ 保存为 SVG
        fname = f"State{s}_{label}.svg"
        fig.savefig(os.path.join(output_dir, fname), format="svg", dpi=300)
        plt.close(fig)                  # 释放内存
        print(f"Saved: {os.path.join(output_dir, fname)}")

        
plot_radar_each_state_pos_neg(
    state_index_dict=state_index_dict_pre,
    subject_ts_dict=subject_ts_dict_pre,
    network_labels=network_labels,
    network_names=network_names,      
    label="Pre",
    output_dir="/media/kaizhang/001/Project_2/DF/FIg/FIg0506/figs_svg"   # 自定义保存路径
)


In [ ]:
# 雷达图 442 Save
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

def plot_radar_each_state_pos_neg(
    state_index_dict,
    subject_ts_dict,
    network_labels,          # 长度 = 442
    network_names,
    label="Post",
    output_dir="."
):
    os.makedirs(output_dir, exist_ok=True)

    n_net = len(network_names)
    theta = np.linspace(0, 2*np.pi, n_net, endpoint=False)
    angles = np.concatenate([theta, theta[:1]])

    # ---------- 收集每个 state 的 TR ----------
    state_ts = defaultdict(list)
    for subj, task_dict in state_index_dict.items():
        for task, state_seq in task_dict.items():
            ts = subject_ts_dict.get(subj, {}).get(task)
            if ts is None:
                continue
            for tr_idx, st in enumerate(state_seq):
                if tr_idx < len(ts):
                    state_ts[st].append(ts[tr_idx])

    # ---------- 逐 state 绘图 ----------
    for s in sorted(state_ts.keys()):
        if not state_ts[s]:
            continue

        # ✅ 使用完整的 442 脑区平均激活
        state_mean_full = np.mean(state_ts[s], axis=0)  # shape: (442,)
        state_mean = state_mean_full  # <-- 保留全部

        # ⬇️ 聚合到各 network（442 regions 被映射到 9 networks）
        net_means = np.array([
            state_mean[network_labels == k].mean()
            for k in range(n_net)
        ])

        pos_vals = np.clip(net_means, 0, None)
        neg_vals = -np.clip(net_means, None, 0)

        pos_plot = np.concatenate([pos_vals, pos_vals[:1]])
        neg_plot = np.concatenate([neg_vals, neg_vals[:1]])

        r_max = max(pos_plot.max(), neg_plot.max(), 1e-6) * 1.05

        # ---------- 画图 ----------
        fig, ax = plt.subplots(figsize=(5, 5), subplot_kw=dict(polar=True))
        ax.spines["polar"].set_visible(False)
        ax.set_ylim(0, r_max)
        ax.yaxis.grid(True, linestyle="--", color="grey", linewidth=0.6)
        ax.xaxis.grid(False)

        for t in theta:
            ax.plot([t, t], [0, r_max], color="grey", linewidth=1)

        ax.plot(angles, pos_plot, color="red", linewidth=3, label="Positive")
        ax.plot(angles, neg_plot, color="blue", linewidth=3, label="Negative")

        ax.set_xticks(theta)
        ax.set_xticklabels([])  # 不显示标签
        ax.set_yticklabels([])

        # 保存为 SVG
        fname = f"State{s}_{label}.svg"
        fig.savefig(os.path.join(output_dir, fname), format="svg", dpi=300)
        plt.close(fig)
        print(f"Saved: {os.path.join(output_dir, fname)}")
        
plot_radar_each_state_pos_neg(
    state_index_dict=state_index_dict_pre,
    subject_ts_dict=subject_ts_dict_pre,
    network_labels=network_labels,
    network_names=network_names,      
    label="Pre",
    output_dir="/media/kaizhang/001/Project_2/DF/FIg/FIg0519/figs_svg"   # 自定义保存路径
)


In [ ]:
# ---------------- Post-Pre -----------------
import os, numpy as np, matplotlib.pyplot as plt
from collections import defaultdict

def plot_radar_each_state_delta(
    state_index_dict_pre,
    state_index_dict_post,
    subject_ts_dict_pre,
    subject_ts_dict_post,
    network_labels,          # 
    network_names,           # 
    output_dir=".",
    min_samples=5            # 
):
    """
    画 Post-Pre 的网络级平均差值
       • 红：Post > Pre（正差）   • 蓝：Post < Pre（负差）
    """
    os.makedirs(output_dir, exist_ok=True)

    n_net   = len(network_names)
    theta   = np.linspace(0, 2*np.pi, n_net, endpoint=False)
    angles  = np.concatenate([theta, theta[:1]])

    # ---------- 收集 TR ----------
    st_pre, st_post = defaultdict(list), defaultdict(list)
    for subj, task_dict in state_index_dict_pre.items():
        for task, seq in task_dict.items():
            ts = subject_ts_dict_pre.get(subj, {}).get(task)
            if ts is None: continue
            for i, st in enumerate(seq):
                if i < len(ts): st_pre[st].append(ts[i])

    for subj, task_dict in state_index_dict_post.items():
        for task, seq in task_dict.items():
            ts = subject_ts_dict_post.get(subj, {}).get(task)
            if ts is None: continue
            for i, st in enumerate(seq):
                if i < len(ts): st_post[st].append(ts[i])

    # ---------- 逐 state ----------
    for s in sorted(set(st_pre) & set(st_post)):
        pre  = np.asarray(st_pre[s])
        post = np.asarray(st_post[s])
        if pre.shape[0] < min_samples or post.shape[0] < min_samples:
            continue

        # ① region-level Δ = mean(Post) − mean(Pre)
        delta = post.mean(axis=0) - pre.mean(axis=0)           # len = 442
        delta_cortex = delta[:len(network_labels)]             # 

        # ② 聚合到网络
        net_delta = np.array([
            delta_cortex[network_labels == k].mean()
            for k in range(n_net)
        ])

        pos_vals = np.clip(net_delta, 0, None)
        neg_vals = -np.clip(net_delta, None, 0)

        pos_plot = np.concatenate([pos_vals, pos_vals[:1]])
        neg_plot = np.concatenate([neg_vals, neg_vals[:1]])

        r_max = max(pos_plot.max(), neg_plot.max(), 1e-6) * 1.05

        # ③ 画雷达
        fig, ax = plt.subplots(figsize=(5, 5), subplot_kw=dict(polar=True))
        ax.spines["polar"].set_visible(False)
        ax.set_ylim(0, r_max)
        ax.yaxis.grid(True, linestyle="--", color="grey", linewidth=0.6)
        ax.xaxis.grid(False)

        # 黑色径向轴
        for t in theta:
            ax.plot([t, t], [0, r_max], color="grey", linewidth=1)

        ax.plot(angles, pos_plot, color="red",  linewidth=3)   # Post > Pre
        ax.plot(angles, neg_plot, color="blue", linewidth=3)   # Post < Pre

        ax.set_xticks(theta)
        ax.set_xticklabels([])
        ax.set_yticklabels([])

        # ④ 保存
        fname = f"State{s}_Post-Pre.svg"
        fig.savefig(os.path.join(output_dir, fname),
                    format="svg", dpi=300)
        plt.close(fig)
        print(f"Saved: {os.path.join(output_dir, fname)}")

plot_radar_each_state_delta(
    state_index_dict_pre   = state_index_dict_pre,
    state_index_dict_post  = state_index_dict_post,
    subject_ts_dict_pre    = subject_ts_dict_pre,
    subject_ts_dict_post   = subject_ts_dict_post,
    network_labels         = network_labels,
    network_names          = network_names,
    output_dir             = "/media/kaizhang/001/Project_2/DF/FIg/FIg0519/radar_delta_svg"
)

In [ ]:
################# Time series plot################################

In [ ]:
# import matplotlib.pyplot as plt
# import numpy as np
# import matplotlib.colors as mcolors
# import numpy as np
# import matplotlib.pyplot as plt

# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.patches as mpatches

# def plot_state_sequence_comparison(subject_id, state_index_dict_pre, state_index_dict_post, task_name, title_prefix=""):
#     # 错误检查
#     if subject_id not in state_index_dict_pre or task_name not in state_index_dict_pre[subject_id]:
#         raise ValueError(f"{subject_id} 不存在于 pre 数据或任务 {task_name} 不存在。")

#     if subject_id not in state_index_dict_post or task_name not in state_index_dict_post[subject_id]:
#         raise ValueError(f"{subject_id} 不存在于 post 数据或任务 {task_name} 不存在。")

#     # 获取状态序列
#     states_pre = np.array(state_index_dict_pre[subject_id][task_name])
#     states_post = np.array(state_index_dict_post[subject_id][task_name])

#     n_tr_pre = len(states_pre)
#     n_tr_post = len(states_post)

#     # 最大状态值（用于颜色范围）
#     max_state = max(states_pre.max(), states_post.max())

#     # 设置绘图
#     fig, axs = plt.subplots(2, 1, figsize=(14, 4), sharex=False)

#     # 使用 tab10 colormap
#     cmap = plt.get_cmap("tab10")

#     axs[0].imshow(states_pre[np.newaxis, :], aspect="auto", cmap=cmap, vmin=0, vmax=max_state)
#     axs[0].set_title(f"{title_prefix} {subject_id} {task_name} - Pre", fontsize=12)
#     axs[0].set_yticks([])

#     axs[1].imshow(states_post[np.newaxis, :], aspect="auto", cmap=cmap, vmin=0, vmax=max_state)
#     axs[1].set_title(f"{title_prefix} {subject_id} {task_name} - Post", fontsize=12)
#     axs[1].set_yticks([])
#     axs[1].set_xlabel("Time (TR)", fontsize=11)

#     # 添加 Legend
#     patches = []
#     for state_num in range(max_state + 1):
#         color = cmap(state_num / (max_state + 0.0001))  # 避免除0
#         patch = mpatches.Patch(color=color, label=f"State {state_num}")
#         patches.append(patch)

#     # Legend放在图外
#     fig.legend(handles=patches, loc='upper center', bbox_to_anchor=(0.5, -0.05),
#                fancybox=True, shadow=True, ncol=min(5, max_state+1), fontsize=10)

#     plt.tight_layout()
#     plt.show()



# ## run
# plot_state_sequence_comparison(
#     subject_id="sub-PCS006",
#     state_index_dict_pre=state_index_dict_pre,
#     state_index_dict_post=state_index_dict_post,
#     task_name="EFIT1",
#     title_prefix="PT"
# )

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# def plot_single_state_occurrence(subject_id,
#                                  state_index_dict_pre,
#                                  state_index_dict_post,
#                                  task_name,
#                                  target_state,
#                                  title_prefix=""):
#     # 检查输入
#     if subject_id not in state_index_dict_pre or task_name not in state_index_dict_pre[subject_id]:
#         raise ValueError(f"{subject_id} 不存在于 pre 数据或任务 {task_name} 不存在。")
#     if subject_id not in state_index_dict_post or task_name not in state_index_dict_post[subject_id]:
#         raise ValueError(f"{subject_id} 不存在于 post 数据或任务 {task_name} 不存在。")

#     # 原始状态序列
#     states_pre = np.array(state_index_dict_pre[subject_id][task_name])
#     states_post = np.array(state_index_dict_post[subject_id][task_name])

#     # 只关注某一个状态：二值化为 [1 if == target_state else 0]
#     binary_pre = (states_pre == target_state).astype(int)
#     binary_post = (states_post == target_state).astype(int)

#     # 状态变化向量（是否进入/离开目标状态）
#     transition_pre = np.concatenate([[0], np.diff(binary_pre)])
#     transition_post = np.concatenate([[0], np.diff(binary_post)])

#     # 绘图
#     fig, axs = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

#     axs[0].plot(binary_pre, color='tab:blue', drawstyle='steps-post')
#     axs[0].set_title(f"{title_prefix} {subject_id} {task_name} - Pre: State {target_state} Presence")
#     axs[0].set_ylabel("In State")

#     axs[1].plot(binary_post, color='tab:green', drawstyle='steps-post')
#     axs[1].set_title(f"{title_prefix} {subject_id} {task_name} - Post: State {target_state} Presence")
#     axs[1].set_ylabel("In State")

#     plt.tight_layout()
#     plt.show()
    
# plot_single_state_occurrence(
#     subject_id="sub-PCS006",
#     state_index_dict_pre=state_index_dict_pre,
#     state_index_dict_post=state_index_dict_post,
#     task_name="EFIT1",
#     target_state=3,
#     title_prefix="PT"
# )


In [ ]:
## For all subject
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.colors import ListedColormap

# ## X-SUB Y-Tr
# def plot_all_subjects_state_matrix(state_index_dict_pre, state_index_dict_post, task_name, title_prefix=""):
#     shared_subjects = []

#     # 找出两个阶段都有该任务的受试者
#     for subj in state_index_dict_pre:
#         if task_name in state_index_dict_pre[subj] and subj in state_index_dict_post:
#             if task_name in state_index_dict_post[subj]:
#                 shared_subjects.append(subj)

#     print(f"共有 {len(shared_subjects)} 名受试者在 Pre/Post 都完成了任务 {task_name}")

#     pre_seqs = []
#     post_seqs = []

#     max_len = 0
#     for subj in shared_subjects:
#         pre = np.array(state_index_dict_pre[subj][task_name])
#         post = np.array(state_index_dict_post[subj][task_name])
#         min_len = min(len(pre), len(post))
#         # 截断到相同长度
#         pre_seqs.append(pre[:min_len])
#         post_seqs.append(post[:min_len])
#         max_len = max(max_len, min_len)

#     # 统一为 (n_subjects, time_points)，然后转置为 time × subject
#     pre_matrix = np.array(pre_seqs).T  # shape: (TR, subjects)
#     post_matrix = np.array(post_seqs).T

#     max_state = max(pre_matrix.max(), post_matrix.max())
#     cmap = plt.get_cmap("tab10")

#     # 画图
#     fig, axs = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

#     axs[0].imshow(pre_matrix, aspect="auto", cmap=cmap, vmin=0, vmax=max_state)
#     axs[0].set_title(f"{title_prefix} All Subjects - Pre - {task_name}", fontsize=12)
#     axs[0].set_ylabel("Time (TR)")

#     axs[1].imshow(post_matrix, aspect="auto", cmap=cmap, vmin=0, vmax=max_state)
#     axs[1].set_title(f"{title_prefix} All Subjects - Post - {task_name}", fontsize=12)
#     axs[1].set_ylabel("Time (TR)")
#     axs[1].set_xlabel("Subject Index")

#     # legend
#     patches = []
#     for state_num in range(max_state + 1):
#         color = cmap(state_num / (max_state + 0.0001))
#         patch = mpatches.Patch(color=color, label=f"State {state_num}")
#         patches.append(patch)

#     fig.legend(handles=patches, loc='upper center', bbox_to_anchor=(0.5, -0.05),
#                fancybox=True, shadow=True, ncol=min(5, max_state+1), fontsize=10)

#     plt.tight_layout()
#     plt.show()

# RUN
# plot_all_subjects_state_matrix(
#     state_index_dict_pre=state_index_dict_pre,
#     state_index_dict_post=state_index_dict_post,
#     task_name="EFIT1",
#     title_prefix="PT"
# )

## Y-SUB X-Tr
def plot_all_subjects_state_matrix(subject_index_dict_pre, subject_index_dict_post, task_name, title_prefix=""):
    shared_subjects = []

    for subj in subject_index_dict_pre:
        if task_name in subject_index_dict_pre[subj] and subj in subject_index_dict_post:
            if task_name in subject_index_dict_post[subj]:
                shared_subjects.append(subj)

    print(f"共有 {len(shared_subjects)} 名受试者在 Pre/Post 都完成了任务 {task_name}")

    pre_seqs = []
    post_seqs = []

    min_tr = float('inf')
    for subj in shared_subjects:
        pre = np.array(subject_index_dict_pre[subj][task_name])
        post = np.array(subject_index_dict_post[subj][task_name])
        # 截断为最短长度
        min_len = min(len(pre), len(post))
        min_tr = min(min_tr, min_len)
        pre_seqs.append(pre[:min_len])
        post_seqs.append(post[:min_len])

    # 组合成矩阵：shape = (subjects, TR)
    pre_matrix = np.array(pre_seqs)  # shape: (n_subjects, TR)
    post_matrix = np.array(post_seqs)

    max_state = max(pre_matrix.max(), post_matrix.max())
    cmap = plt.get_cmap("tab10")

    fig, axs = plt.subplots(2, 1, figsize=(18, 10), sharex=True)

    axs[0].imshow(pre_matrix, aspect="auto", cmap=cmap, vmin=0, vmax=max_state)
    axs[0].set_title(f"{title_prefix} All Subjects - Pre - {task_name}", fontsize=14)
    axs[0].set_ylabel("Subjects", fontsize=12)

    axs[1].imshow(post_matrix, aspect="auto", cmap=cmap, vmin=0, vmax=max_state)
    axs[1].set_title(f"{title_prefix} All Subjects - Post - {task_name}", fontsize=14)
    axs[1].set_ylabel("Subjects", fontsize=12)
    axs[1].set_xlabel("Time (TR)", fontsize=12)

    # 添加 legend
    patches = []
    for state_num in range(max_state + 1):
        color = cmap(state_num / (max_state + 0.0001))
        patch = mpatches.Patch(color=color, label=f"State {state_num}")
        patches.append(patch)

    fig.legend(handles=patches, loc='upper center', bbox_to_anchor=(0.5, -0.03),
               fancybox=True, shadow=True, ncol=min(5, max_state+1), fontsize=11)

    plt.tight_layout()
    plt.show()
    
# RUN
plot_all_subjects_state_matrix(
    subject_index_dict_pre=state_index_dict_pre,
    subject_index_dict_post=state_index_dict_post,
    task_name="EFIT2",
    title_prefix="PT"
)

In [ ]:
# For one state
def plot_all_subjects_state_matrix(subject_index_dict_pre, subject_index_dict_post,
                                   task_name, title_prefix="", target_state=None):
    shared_subjects = []

    for subj in subject_index_dict_pre:
        if task_name in subject_index_dict_pre[subj] and subj in subject_index_dict_post:
            if task_name in subject_index_dict_post[subj]:
                shared_subjects.append(subj)

    print(f"共有 {len(shared_subjects)} 名受试者在 Pre/Post 都完成了任务 {task_name}")

    pre_seqs = []
    post_seqs = []
    min_tr = float('inf')

    for subj in shared_subjects:
        pre = np.array(subject_index_dict_pre[subj][task_name])
        post = np.array(subject_index_dict_post[subj][task_name])
        min_len = min(len(pre), len(post))
        min_tr = min(min_tr, min_len)
        pre_seqs.append(pre[:min_len])
        post_seqs.append(post[:min_len])

    pre_matrix = np.array(pre_seqs)
    post_matrix = np.array(post_seqs)

    max_state = max(pre_matrix.max(), post_matrix.max())

    # 只保留目标状态
    if target_state is not None:
        pre_matrix = np.where(pre_matrix == target_state, target_state, np.nan)
        post_matrix = np.where(post_matrix == target_state, target_state, np.nan)
        cmap = ListedColormap(['blue'])  # 只需要一个色
    else:
        palette = sns.color_palette("Set2", n_colors=max_state + 1)
        cmap = ListedColormap(palette)

    #  绘图
    fig, axs = plt.subplots(2, 1, figsize=(18, 10), sharex=True)

    axs[0].imshow(pre_matrix, aspect="auto", cmap=cmap, vmin=0, vmax=max_state)
    axs[0].set_title(f"{title_prefix} All Subjects - Pre - {task_name}", fontsize=14)
    axs[0].set_ylabel("Subjects", fontsize=12)

    axs[1].imshow(post_matrix, aspect="auto", cmap=cmap, vmin=0, vmax=max_state)
    axs[1].set_title(f"{title_prefix} All Subjects - Post - {task_name}", fontsize=14)
    axs[1].set_ylabel("Subjects", fontsize=12)
    axs[1].set_xlabel("Time (TR)", fontsize=12)

    # legend（仅当绘制所有状态时保留 legend）
    if target_state is None:
        patches = []
        for state_num in range(max_state + 1):
            color = cmap(state_num / (max_state + 0.0001))
            patch = mpatches.Patch(color=color, label=f"State {state_num}")
            patches.append(patch)
        fig.legend(handles=patches, loc='upper center', bbox_to_anchor=(0.5, -0.03),
                   fancybox=True, shadow=True, ncol=min(5, max_state+1), fontsize=11)

    plt.tight_layout()
    plt.show()

plot_all_subjects_state_matrix(
    subject_index_dict_pre=state_index_dict_pre,
    subject_index_dict_post=state_index_dict_post,
    task_name="REST",
    title_prefix="PT",
    target_state=6
)


In [ ]:
## ERT PLOT（Length unique）
def plot_all_subjects_state_matrix(subject_index_dict_pre, subject_index_dict_post, task_name, title_prefix=""):
    shared_subjects = []

    for subj in subject_index_dict_pre:
        if task_name in subject_index_dict_pre[subj] and subj in subject_index_dict_post:
            if task_name in subject_index_dict_post[subj]:
                shared_subjects.append(subj)

    print(f"初步找到 {len(shared_subjects)} 名受试者在 Pre/Post 都完成了任务 {task_name}")

    # 第一步：找出最短的可用 TR 数
    lengths = []
    for subj in shared_subjects:
        pre_len = len(subject_index_dict_pre[subj][task_name])
        post_len = len(subject_index_dict_post[subj][task_name])
        lengths.append(min(pre_len, post_len))

    if not lengths:
        raise ValueError("没有找到有效的受试者用于统一长度截断。")

    min_len = min(lengths)
    print(f"统一截断长度（TR）为: {min_len}")

    pre_seqs = []
    post_seqs = []
    actual_subjects = []

    for subj in shared_subjects:
        pre = np.array(subject_index_dict_pre[subj][task_name])
        post = np.array(subject_index_dict_post[subj][task_name])

        if len(pre) >= min_len and len(post) >= min_len:
            pre_seqs.append(pre[:min_len])
            post_seqs.append(post[:min_len])
            actual_subjects.append(subj)

    # 转换为矩阵 (subjects × TR)
    pre_matrix = np.array(pre_seqs)
    post_matrix = np.array(post_seqs)

    max_state = max(pre_matrix.max(), post_matrix.max())
    cmap = plt.get_cmap("tab10")

    fig, axs = plt.subplots(2, 1, figsize=(18, 10), sharex=True)

    axs[0].imshow(pre_matrix, aspect="auto", cmap=cmap, vmin=0, vmax=max_state)
    axs[0].set_title(f"{title_prefix} All Subjects - Pre - {task_name}", fontsize=14)
    axs[0].set_ylabel("Subjects", fontsize=12)

    axs[1].imshow(post_matrix, aspect="auto", cmap=cmap, vmin=0, vmax=max_state)
    axs[1].set_title(f"{title_prefix} All Subjects - Post - {task_name}", fontsize=14)
    axs[1].set_ylabel("Subjects", fontsize=12)
    axs[1].set_xlabel("Time (TR)", fontsize=12)

    # Legend
    import matplotlib.patches as mpatches
    patches = []
    for state_num in range(max_state + 1):
        color = cmap(state_num / (max_state + 0.0001))
        patch = mpatches.Patch(color=color, label=f"State {state_num}")
        patches.append(patch)

    fig.legend(handles=patches, loc='upper center', bbox_to_anchor=(0.5, -0.03),
               fancybox=True, shadow=True, ncol=min(5, max_state+1), fontsize=11)

    plt.tight_layout()
    plt.show()

    
plot_all_subjects_state_matrix(
    subject_index_dict_pre=state_index_dict_pre,
    subject_index_dict_post=state_index_dict_post,
    task_name="ERT2",  
    title_prefix="PT"
)
